# 🎓 Smart University AI FAQ Chatbot — Gradio Edition (1,000-Question Dataset)

A full-featured AI FAQ assistant, trained on your **1,000-question university
FAQ dataset**, with a real web GUI built using **Gradio** — runs entirely
inside this one Colab notebook.

**Matching pipeline:** Text Cleaning → Tokenization → Stopword Removal +
Lemmatization → TF-IDF Vectorization → Cosine Similarity → Best Answer.

```
              UNIVERSITY AI CHATBOT
                       │
   ┌──────┬──────┬─────┼─────┬──────┬──────┐
   ↓      ↓      ↓     ↓     ↓      ↓      ↓
Admission Programs Fees Exams Hostel Library  ...(16 categories total)
                       │
                       ↓
                 NLP Processing
                       ↓
                TF-IDF + Cosine
                  Similarity
                       ↓
                Best Answer
```

**Dataset:** 1,000 questions across 16 categories — Admission, Programs,
Fees, Exams, Attendance, Scholarships, Library, Hostel, Student Portal,
Courses, Faculty, Campus Facilities, Graduation, Rules & Policies,
Timetable, General Student Services.

**Features**
- 💬 Chat interface (`gr.Chatbot`) with separate user/bot bubbles
- 📂 16 FAQ categories, with a filter that also scopes matching
- 💡 Suggested questions (clickable, filterable by category)
- 📊 Confidence score + matched category shown per answer
- 🤔 Fallback message + human-support note when confidence is too low
- 🗑️ Chat history with a Clear button
- 🛠️ Admin panel: search, add, edit, delete FAQs; view all in a table;
  save to disk or download the updated dataset
- 🔊 Text-to-speech (bot reads the answer aloud, via gTTS)
- 🎤 Voice input (record a question with your mic, transcribed via
  SpeechRecognition)
- 🌙 Dark / light mode toggle

Run every cell top to bottom. The last cell launches the app and prints a
public link (Gradio's `share=True`, since Colab can't serve `localhost`
directly to your browser) — click it to open the GUI.


## 1. Install dependencies

In [ ]:
!pip install -q gradio scikit-learn nltk pandas SpeechRecognition gTTS


In [ ]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
print("NLTK data ready ✅")


## 2. Your dataset

**Option A (default, used below):** the 1,000-question dataset you provided
is embedded in the next cell and written straight to `faq.csv`.

**Option B:** if you'd rather upload a different/updated CSV yourself, run
this cell instead — it lets you pick a file from your computer, and skips
writing the embedded version.


In [ ]:
# Uncomment to upload your own CSV instead of using the embedded dataset below.
# Must have columns: id, category, question, answer

# from google.colab import files
# uploaded = files.upload()
# import shutil
# csv_name = list(uploaded.keys())[0]
# shutil.move(csv_name, "faq.csv")
# print(f"Using uploaded file: {csv_name}")


### `faq.csv` — your 1,000-question dataset (embedded)

In [ ]:
%%writefile faq.csv
﻿id,category,question,answer
1,Admission,What are the admission requirements for BS Computer Science?,Admission requirements for BS Computer Science depend on the university's current eligibility policy. Please check the official admissions requirements.
2,Admission,What are the admission requirements for BS Artificial Intelligence?,Admission requirements for BS Artificial Intelligence depend on the university's current eligibility policy. Please check the official admissions requirements.
3,Admission,What are the admission requirements for BS Software Engineering?,Admission requirements for BS Software Engineering depend on the university's current eligibility policy. Please check the official admissions requirements.
4,Admission,What are the admission requirements for BS Information Technology?,Admission requirements for BS Information Technology depend on the university's current eligibility policy. Please check the official admissions requirements.
5,Admission,What are the admission requirements for BBA?,Admission requirements for BBA depend on the university's current eligibility policy. Please check the official admissions requirements.
6,Admission,What are the admission requirements for MBA?,Admission requirements for MBA depend on the university's current eligibility policy. Please check the official admissions requirements.
7,Admission,How can I apply for admission to BS Computer Science?,You can apply for BS Computer Science through the university's official admission portal or admissions office.
8,Admission,How can I apply for admission to BS Artificial Intelligence?,You can apply for BS Artificial Intelligence through the university's official admission portal or admissions office.
9,Admission,How can I apply for admission to BS Software Engineering?,You can apply for BS Software Engineering through the university's official admission portal or admissions office.
10,Admission,How can I apply for admission to BS Information Technology?,You can apply for BS Information Technology through the university's official admission portal or admissions office.
11,Admission,How can I apply for admission to BBA?,You can apply for BBA through the university's official admission portal or admissions office.
12,Admission,How can I apply for admission to MBA?,You can apply for MBA through the university's official admission portal or admissions office.
13,Admission,When does admission open for BS Computer Science?,Admission opening dates vary by university and intake. Please check the latest official admission schedule.
14,Admission,When does admission open for BS Artificial Intelligence?,Admission opening dates vary by university and intake. Please check the latest official admission schedule.
15,Admission,When does admission open for BS Software Engineering?,Admission opening dates vary by university and intake. Please check the latest official admission schedule.
16,Admission,When does admission open for BS Information Technology?,Admission opening dates vary by university and intake. Please check the latest official admission schedule.
17,Admission,When does admission open for BBA?,Admission opening dates vary by university and intake. Please check the latest official admission schedule.
18,Admission,When does admission open for MBA?,Admission opening dates vary by university and intake. Please check the latest official admission schedule.
19,Admission,What documents are required for admission?,"Common documents include academic certificates, identification documents, photographs, and the completed application form. The university may require additional documents."
20,Admission,Can I apply online for admission?,Most universities provide an online admission application system. Please use the official university admission portal.
21,Admission,What is the admission deadline?,The admission deadline depends on the program and intake. Please check the current official admission notice.
22,Admission,Can I apply after the deadline?,Late applications are accepted only if the university announces an extension. Check the latest official notice.
23,Admission,How can I check my admission application status?,Check your application status through the university admission portal or contact the admissions office.
24,Admission,What is the admission fee?,The admission or application fee varies by university and program. Check the current fee schedule.
25,Admission,Can I change my selected program after applying?,Program changes may be possible before a specified deadline. Contact the admissions office for the current procedure.
26,Programs,What programs does the university offer?,The university offers programs listed in its current academic prospectus. Check the official programs and departments list.
27,Programs,Does the university offer BS Computer Science?,Please check the university's current programs list to confirm whether BS Computer Science is offered.
28,Programs,Does the university offer BS Artificial Intelligence?,Please check the university's current programs list to confirm whether BS Artificial Intelligence is offered.
29,Programs,Does the university offer BS Software Engineering?,Please check the university's current programs list to confirm whether BS Software Engineering is offered.
30,Programs,Does the university offer BS Information Technology?,Please check the university's current programs list to confirm whether BS Information Technology is offered.
31,Programs,Does the university offer BBA?,Please check the university's current programs list to confirm whether BBA is offered.
32,Programs,Does the university offer MBA?,Please check the university's current programs list to confirm whether MBA is offered.
33,Programs,How long is the BS Computer Science program?,The duration of BS Computer Science depends on the university's approved curriculum and academic regulations.
34,Programs,How long is the BS Artificial Intelligence program?,The duration of BS Artificial Intelligence depends on the university's approved curriculum and academic regulations.
35,Programs,How long is the BS Software Engineering program?,The duration of BS Software Engineering depends on the university's approved curriculum and academic regulations.
36,Programs,How long is the BS Information Technology program?,The duration of BS Information Technology depends on the university's approved curriculum and academic regulations.
37,Programs,How long is the BBA program?,The duration of BBA depends on the university's approved curriculum and academic regulations.
38,Programs,How long is the MBA program?,The duration of MBA depends on the university's approved curriculum and academic regulations.
39,Programs,What subjects are included in BS Computer Science?,The subjects for BS Computer Science are listed in the official curriculum or degree plan.
40,Programs,What subjects are included in BS Artificial Intelligence?,The subjects for BS Artificial Intelligence are listed in the official curriculum or degree plan.
41,Programs,What subjects are included in BS Software Engineering?,The subjects for BS Software Engineering are listed in the official curriculum or degree plan.
42,Programs,What subjects are included in BS Information Technology?,The subjects for BS Information Technology are listed in the official curriculum or degree plan.
43,Programs,What subjects are included in BBA?,The subjects for BBA are listed in the official curriculum or degree plan.
44,Programs,What subjects are included in MBA?,The subjects for MBA are listed in the official curriculum or degree plan.
45,Programs,Is BS Computer Science available in morning classes?,Class timing depends on the university and department schedule. Check the current timetable.
46,Programs,Is BS Artificial Intelligence available in morning classes?,Class timing depends on the university and department schedule. Check the current timetable.
47,Programs,Is BS Software Engineering available in morning classes?,Class timing depends on the university and department schedule. Check the current timetable.
48,Programs,Is BS Information Technology available in morning classes?,Class timing depends on the university and department schedule. Check the current timetable.
49,Programs,Is BBA available in morning classes?,Class timing depends on the university and department schedule. Check the current timetable.
50,Programs,Is MBA available in morning classes?,Class timing depends on the university and department schedule. Check the current timetable.
51,Programs,Is BS Computer Science available in evening classes?,Evening availability depends on the department's current schedule.
52,Programs,Is BS Artificial Intelligence available in evening classes?,Evening availability depends on the department's current schedule.
53,Programs,Is BS Software Engineering available in evening classes?,Evening availability depends on the department's current schedule.
54,Programs,Is BS Information Technology available in evening classes?,Evening availability depends on the department's current schedule.
55,Programs,Is BBA available in evening classes?,Evening availability depends on the department's current schedule.
56,Programs,Is MBA available in evening classes?,Evening availability depends on the department's current schedule.
57,Programs,What is the scope of BS Computer Science?,"The scope of BS Computer Science depends on industry demand, skills, specialization, and career opportunities."
58,Programs,What is the scope of BS Artificial Intelligence?,"The scope of BS Artificial Intelligence depends on industry demand, skills, specialization, and career opportunities."
59,Programs,What is the scope of BS Software Engineering?,"The scope of BS Software Engineering depends on industry demand, skills, specialization, and career opportunities."
60,Programs,What is the scope of BS Information Technology?,"The scope of BS Information Technology depends on industry demand, skills, specialization, and career opportunities."
61,Programs,What is the scope of BBA?,"The scope of BBA depends on industry demand, skills, specialization, and career opportunities."
62,Programs,What is the scope of MBA?,"The scope of MBA depends on industry demand, skills, specialization, and career opportunities."
63,Programs,What careers are available after BS Computer Science?,Career options after BS Computer Science depend on the skills and specialization you develop. Consult the program's official career information.
64,Programs,What careers are available after BS Artificial Intelligence?,Career options after BS Artificial Intelligence depend on the skills and specialization you develop. Consult the program's official career information.
65,Programs,What careers are available after BS Software Engineering?,Career options after BS Software Engineering depend on the skills and specialization you develop. Consult the program's official career information.
66,Programs,What careers are available after BS Information Technology?,Career options after BS Information Technology depend on the skills and specialization you develop. Consult the program's official career information.
67,Programs,What careers are available after BBA?,Career options after BBA depend on the skills and specialization you develop. Consult the program's official career information.
68,Programs,What careers are available after MBA?,Career options after MBA depend on the skills and specialization you develop. Consult the program's official career information.
69,Programs,Can I change my major?,Major or program changes are subject to university academic regulations and available seats.
70,Programs,Where can I find the degree curriculum?,"The official curriculum can usually be found on the university website, department office, or academic portal."
71,Fees,What is the tuition fee for BS Computer Science?,The tuition fee for BS Computer Science should be confirmed from the university's latest official fee structure.
72,Fees,What is the tuition fee for BS Artificial Intelligence?,The tuition fee for BS Artificial Intelligence should be confirmed from the university's latest official fee structure.
73,Fees,What is the tuition fee for BS Software Engineering?,The tuition fee for BS Software Engineering should be confirmed from the university's latest official fee structure.
74,Fees,What is the tuition fee for BS Information Technology?,The tuition fee for BS Information Technology should be confirmed from the university's latest official fee structure.
75,Fees,What is the tuition fee for BBA?,The tuition fee for BBA should be confirmed from the university's latest official fee structure.
76,Fees,What is the tuition fee for MBA?,The tuition fee for MBA should be confirmed from the university's latest official fee structure.
77,Fees,How can I pay my semester fee?,"Use the payment methods authorized by the university, such as its online portal or designated bank/payment channels."
78,Fees,What is the last date for fee payment?,The fee payment deadline is published in the academic calendar or fee notice.
79,Fees,Is there a late fee for delayed payment?,A late payment charge may apply according to university regulations. Check the latest fee policy.
80,Fees,Can I pay my fee in installments?,Installment options depend on university policy and may require approval from the relevant office.
81,Fees,How can I get my fee voucher?,Fee vouchers are normally available through the student portal or accounts office.
82,Fees,What should I do if my fee payment is not updated?,Keep your payment receipt and contact the accounts or finance office to have the transaction verified.
83,Fees,Can I get a fee refund?,Refund eligibility depends on the university's refund policy and the reason for withdrawal or cancellation.
84,Fees,How can I request a fee refund?,Submit a refund request through the procedure specified by the university finance or accounts office.
85,Fees,Where can I find the latest fee structure?,The latest fee structure should be obtained from the official university website or accounts office.
86,Exams,When will the midterm exams start?,Midterm dates are published in the university's academic calendar or examination schedule.
87,Exams,When will the final exams start?,Final examination dates are announced by the examination office and academic calendar.
88,Exams,Where can I find the exam timetable?,"Check the student portal, examination office notice board, or official university announcements."
89,Exams,What should I bring to the exam?,Bring the identification and stationery permitted by the university examination rules.
90,Exams,What happens if I miss an exam?,"If you miss an exam, follow the university's rules for absence and contact the examination office promptly."
91,Exams,Can I apply for a re-examination?,Re-examination opportunities depend on the university's examination regulations.
92,Exams,How can I apply for a supplementary exam?,Apply through the examination office according to the current supplementary examination procedure.
93,Exams,What are the examination rules?,Examination rules are defined by the university examination regulations and should be followed by all students.
94,Exams,When are exam results announced?,Results are announced after evaluation and approval according to the university's examination process.
95,Exams,How can I check my exam result?,Check your result through the official student portal or examination office.
96,Attendance,What is the minimum attendance requirement?,The minimum attendance requirement is defined by the university's academic regulations. Check the current attendance policy.
97,Attendance,How can I check my attendance?,Attendance can usually be viewed through the student portal or confirmed by the relevant department.
98,Attendance,What happens if my attendance is below the required percentage?,Students below the required attendance level may face academic restrictions according to university policy.
99,Attendance,Can I request attendance correction?,"If your attendance record is incorrect, contact the course instructor or department office with supporting evidence."
100,Attendance,Does medical leave affect attendance?,Medical leave is handled according to university leave and attendance regulations.
101,Attendance,How do I apply for academic leave?,Submit a leave application through the procedure specified by your department or student affairs office.
102,Attendance,Can attendance be marked late?,Late attendance rules depend on the instructor and university attendance policy.
103,Attendance,Who maintains the attendance record?,Attendance is normally recorded by course instructors and maintained through the department or academic system.
104,Attendance,Can I attend another section to make up attendance?,Section changes or make-up attendance are allowed only when approved under university policy.
105,Attendance,Why is my attendance missing from the portal?,Contact the course instructor or department if attendance has not been updated or appears incorrect.
106,Scholarships,What scholarships are available for students?,"Available scholarships depend on the university, government programs, donors, and current scholarship announcements."
107,Scholarships,How can I apply for a scholarship?,Follow the application procedure published by the university scholarship or financial aid office.
108,Scholarships,What are the scholarship eligibility criteria?,"Eligibility varies by scholarship and may consider academic performance, financial need, or other criteria."
109,Scholarships,When is the scholarship application deadline?,Check the current scholarship announcement for the exact deadline.
110,Scholarships,Is there a merit scholarship?,"Many universities offer merit-based scholarships, but availability and criteria should be confirmed from the current official notice."
111,Scholarships,Is financial assistance available?,"Financial assistance may be available through scholarships, aid programs, or fee support. Contact the financial aid office."
112,Scholarships,Can international students apply for scholarships?,Eligibility depends on the scholarship rules and university policy.
113,Scholarships,Can I receive more than one scholarship?,Receiving multiple scholarships depends on the terms and conditions of the specific programs.
114,Scholarships,How will I know if my scholarship is approved?,"Scholarship decisions are normally communicated through the student portal, official notice, or scholarship office."
115,Scholarships,Who should I contact about scholarships?,Contact the university scholarship or financial aid office for current information.
116,Library,What are the library opening hours?,Library hours vary by campus and semester. Check the latest library schedule.
117,Library,How can I get a library card?,Library membership is normally activated through the university library according to its registration procedure.
118,Library,How many books can I borrow?,The borrowing limit depends on student status and library policy.
119,Library,How long can I keep a borrowed book?,The loan period is determined by the library's current borrowing policy.
120,Library,Can I renew a borrowed book?,Books may be renewable if they are eligible and not reserved by another user.
121,Library,What happens if I return a book late?,Late returns may result in fines or borrowing restrictions according to library policy.
122,Library,How can I search for a book?,Use the university library catalog or ask library staff for assistance.
123,Library,Can I access online journals?,"If the university subscribes to online databases, students can access them through the library's authorized services."
124,Library,Can I reserve a book?,Book reservation availability depends on the library system and current policy.
125,Library,Where is the university library?,The library location depends on the campus. Check the official campus map or library information.
126,Hostel,How can I apply for university hostel accommodation?,Submit a hostel application through the university's approved hostel application procedure.
127,Hostel,Who is eligible for hostel accommodation?,"Eligibility depends on university hostel rules, available rooms, and student status."
128,Hostel,What is the hostel fee?,Hostel fees vary by campus and room type. Check the latest official hostel fee schedule.
129,Hostel,When is hostel admission open?,Hostel application dates are announced by the university hostel administration.
130,Hostel,Can I choose my roommate?,Roommate selection depends on hostel policy and room availability.
131,Hostel,What documents are required for hostel admission?,"Required documents may include student identification, admission proof, photographs, and other documents specified by hostel administration."
132,Hostel,What are the hostel rules?,"Hostel rules cover residence, discipline, visitors, safety, and other matters. Follow the current official hostel regulations."
133,Hostel,Can I leave the hostel during the semester?,Hostel withdrawal or room cancellation follows the hostel administration's procedure.
134,Hostel,How can I report a hostel problem?,"Report hostel maintenance, safety, or administrative issues to the hostel warden or designated office."
135,Hostel,Are meals provided in the hostel?,Meal facilities depend on the hostel and campus arrangements. Check the current hostel information.
136,Student Portal,How do I log in to the student portal?,Use the credentials provided by the university and access the official student portal.
137,Student Portal,I forgot my student portal password. What should I do?,Use the portal's password recovery option or contact the university IT/help desk.
138,Student Portal,How can I change my portal password?,Use the account settings or password-change option in the official student portal.
139,Student Portal,How can I update my profile information?,"Update profile information through the student portal if editing is enabled, or contact the relevant office."
140,Student Portal,Why can't I log in to the student portal?,"Check your credentials and internet connection. If the problem continues, contact the university IT support office."
141,Student Portal,How can I register for courses online?,Use the course registration feature in the student portal during the announced registration period.
142,Student Portal,How can I download my fee voucher?,Log in to the student portal and use the fee or finance section if the university provides electronic vouchers.
143,Student Portal,How can I download my transcript?,"If electronic transcripts are supported, request or download them through the student portal; otherwise contact the registrar."
144,Student Portal,How can I see my registered courses?,Open the registered courses or enrollment section of the student portal.
145,Student Portal,How can I contact technical support for the portal?,Contact the university IT help desk through the official support channel.
146,Courses,How do I register for a course?,Course registration is completed through the university's registration system during the announced registration period.
147,Courses,Can I drop a course?,Course withdrawal is allowed only within the period and conditions defined by academic regulations.
148,Courses,Can I add a course after registration?,Adding a course after registration depends on the add/drop deadline and departmental approval.
149,Courses,What is a prerequisite course?,A prerequisite is a course that must normally be completed before taking another course.
150,Courses,How can I find my course prerequisites?,Check the official degree plan or course catalog for prerequisite requirements.
151,Courses,Can I repeat a failed course?,Course repetition is generally governed by the university's academic regulations.
152,Courses,What is a credit hour?,A credit hour is a unit used to measure the academic workload of a course.
153,Courses,How many courses can I take in a semester?,"The allowed course load depends on the program, semester, and university academic regulations."
154,Courses,Can I take an extra course?,An extra course may require academic approval and must comply with the maximum credit-hour policy.
155,Courses,What happens if I withdraw from a course?,The academic and transcript consequences depend on the university's withdrawal policy.
156,Faculty,How can I contact my course instructor?,"Use the official university email, learning platform, department office, or other approved communication channel."
157,Faculty,Where can I find faculty office hours?,"Faculty office hours may be listed on the department website, student portal, or department notice board."
158,Faculty,Who is the head of my department?,The current department head should be confirmed from the official university department page.
159,Faculty,How can I meet a faculty member?,Contact the faculty member through the official channel and request an appointment during available hours.
160,Faculty,How can I submit an assignment to my instructor?,Submit assignments using the platform or method specified by the instructor.
161,Faculty,What should I do if I have an academic issue with a course?,"First discuss the issue with the course instructor, then follow the department's academic complaint procedure if necessary."
162,Faculty,How can I request a recommendation letter?,Ask the faculty member according to the university's recommendation or reference procedure and provide the required information.
163,Faculty,Can I change my academic advisor?,Advisor changes depend on departmental policy and approval.
164,Faculty,Who is my academic advisor?,Your academic advisor can be identified through the student portal or department office.
165,Faculty,How can I contact the department office?,Use the department's official contact information listed on the university website.
166,Campus Facilities,Does the university have a computer lab?,Check the campus facilities list to confirm available computer labs and their locations.
167,Campus Facilities,Does the university provide Wi-Fi?,University Wi-Fi availability and access rules depend on the campus IT policy.
168,Campus Facilities,How can I access campus Wi-Fi?,Use the credentials and connection instructions provided by the university IT department.
169,Campus Facilities,Is there a cafeteria on campus?,Cafeteria availability depends on the campus. Check the official campus facilities information.
170,Campus Facilities,Does the campus have a sports facility?,Sports facilities vary by campus and can be confirmed through student affairs or campus facilities information.
171,Campus Facilities,Is there a medical center on campus?,Check the campus facilities or student services information for available medical services.
172,Campus Facilities,Where can I report a maintenance issue?,Report maintenance problems to the campus facilities or administration office through the approved channel.
173,Campus Facilities,Is parking available for students?,Student parking availability and permits depend on campus parking rules.
174,Campus Facilities,How can I get a student ID card?,Student ID cards are issued through the university's designated student services or administration office.
175,Campus Facilities,What should I do if I lose my student ID card?,Report the lost card immediately to student services and follow the replacement procedure.
176,Graduation,What are the graduation requirements?,Graduation requirements are defined by the degree plan and university academic regulations.
177,Graduation,How can I apply for graduation?,Submit a graduation application through the registrar or student portal according to the announced procedure.
178,Graduation,When should I apply for graduation?,Apply during the graduation application period announced by the registrar.
179,Graduation,How can I check whether I have completed my degree requirements?,Review your degree audit or contact your academic advisor or registrar.
180,Graduation,What documents are needed for graduation clearance?,"Required documents vary by university and may include clearance forms, identification, and financial or departmental clearance."
181,Graduation,When is the graduation ceremony?,Ceremony dates are announced by the university through official notices.
182,Graduation,How can I get my degree certificate?,Degree certificates are issued by the registrar or examination authority after completion of all requirements.
183,Graduation,Can I attend graduation if my clearance is incomplete?,Participation depends on the university's graduation and clearance rules.
184,Graduation,How can I request a duplicate degree?,Follow the registrar's procedure for replacement or duplicate degree documents.
185,Graduation,How can I obtain an official transcript after graduation?,Request an official transcript through the registrar or examination office.
186,Rules & Policies,Where can I find university rules?,"University rules are normally published in official regulations, student handbooks, or the university website."
187,Rules & Policies,What is the student code of conduct?,The student code of conduct defines expected academic and disciplinary behavior.
188,Rules & Policies,What happens if a student violates university rules?,Disciplinary action depends on the nature of the violation and the university's disciplinary regulations.
189,Rules & Policies,How can I submit a complaint?,Use the official complaint or grievance procedure provided by student affairs or the relevant office.
190,Rules & Policies,How can I appeal an academic decision?,Follow the university's formal academic appeal procedure within the specified deadline.
191,Rules & Policies,What is the plagiarism policy?,Students must follow the university's academic integrity and plagiarism policy.
192,Rules & Policies,What is considered academic misconduct?,"Academic misconduct may include plagiarism, cheating, unauthorized collaboration, or other violations defined by university regulations."
193,Rules & Policies,Can students use mobile phones during exams?,Mobile phone use during exams is governed by examination rules and may be prohibited.
194,Rules & Policies,What is the dress code?,"Dress requirements, if any, are defined by the university or campus policy."
195,Rules & Policies,Where can I report a disciplinary concern?,"Report concerns through the university's designated student affairs, discipline, or complaint channel."
196,Timetable,Where can I find my class timetable?,"Check the student portal, department notice board, or official timetable announcement."
197,Timetable,How can I know my classroom?,Classroom assignments are usually shown in the timetable or announced by the department.
198,Timetable,Can the timetable change during the semester?,"Yes, timetable changes may occur due to academic or administrative requirements. Check official announcements."
199,Timetable,How can I report a timetable conflict?,Report schedule conflicts to the department or academic office promptly.
200,Timetable,How can I find the timetable for my department?,Check the department's official timetable announcement or student portal.
201,Timetable,What should I do if two classes are scheduled at the same time?,Contact the department or academic office to report the conflict and request guidance.
202,Timetable,Where can I find room numbers?,Room numbers are normally listed with course schedules or posted by the department.
203,Timetable,How do I know if a class has been cancelled?,"Check official university announcements, the student portal, or communication from the instructor."
204,Timetable,Can I change my class section?,Section changes depend on available seats and department approval.
205,Timetable,When is the class schedule published?,The schedule is normally published before the start of the academic term.
206,General Student Services,How can I contact student affairs?,Use the official student affairs office contact details published by the university.
207,General Student Services,Where is the registrar office?,The registrar office location depends on the campus. Check the official campus directory.
208,General Student Services,What does the registrar office handle?,"The registrar commonly handles academic records, enrollment, transcripts, and graduation documentation."
209,General Student Services,How can I request an enrollment certificate?,Request an enrollment certificate through the registrar or student portal according to university procedure.
210,General Student Services,How can I request a bonafide student certificate?,Apply through the registrar or designated student services office.
211,General Student Services,How can I change my personal information?,Submit an update request through the student portal or registrar with required supporting documents.
212,General Student Services,How can I request a transcript?,Submit a transcript request through the registrar or examination office.
213,General Student Services,How can I get an official university letter?,Request the required letter from the relevant administrative office using the university's official procedure.
214,General Student Services,What is the university contact number?,Use the current official contact number published on the university website.
215,General Student Services,What is the university email address?,Use the current official email address published by the university.
216,Admission,What are the admission requirements for BS Computer Science (information for students) for current students?,Admission requirements for BS Computer Science depend on the university's current eligibility policy. Please check the official admissions requirements.
217,Admission,What are the admission requirements for BS Artificial Intelligence (information for students) for current students?,Admission requirements for BS Artificial Intelligence depend on the university's current eligibility policy. Please check the official admissions requirements.
218,Admission,What are the admission requirements for BS Software Engineering (information for students) for current students?,Admission requirements for BS Software Engineering depend on the university's current eligibility policy. Please check the official admissions requirements.
219,Admission,What are the admission requirements for BS Information Technology (information for students) for current students?,Admission requirements for BS Information Technology depend on the university's current eligibility policy. Please check the official admissions requirements.
220,Admission,What are the admission requirements for BBA (information for students) for current students?,Admission requirements for BBA depend on the university's current eligibility policy. Please check the official admissions requirements.
221,Admission,What are the admission requirements for MBA (information for students) for current students?,Admission requirements for MBA depend on the university's current eligibility policy. Please check the official admissions requirements.
222,Admission,How can I apply for admission to BS Computer Science (information for students) for the current academic year?,You can apply for BS Computer Science through the university's official admission portal or admissions office.
223,Admission,What is the procedure to apply for admission to BS Artificial Intelligence for the current academic year?,You can apply for BS Artificial Intelligence through the university's official admission portal or admissions office.
224,Admission,How can I apply for admission to BS Software Engineering (information for students) for the current academic year?,You can apply for BS Software Engineering through the university's official admission portal or admissions office.
225,Admission,How can I apply for admission to BS Information Technology (information for students) for the current academic year?,You can apply for BS Information Technology through the university's official admission portal or admissions office.
226,Admission,How can I apply for admission to BBA (information for students) for the current academic year?,You can apply for BBA through the university's official admission portal or admissions office.
227,Admission,How can I apply for admission to MBA (information for students) for the current academic year?,You can apply for MBA through the university's official admission portal or admissions office.
228,Admission,When does admission open for BS Computer Science (information for students) according to university policy?,Admission opening dates vary by university and intake. Please check the latest official admission schedule.
229,Admission,When does admission open for BS Artificial Intelligence (information for students) according to university policy?,Admission opening dates vary by university and intake. Please check the latest official admission schedule.
230,Admission,When does admission open for BS Software Engineering (information for students) according to university policy?,Admission opening dates vary by university and intake. Please check the latest official admission schedule.
231,Admission,When does admission open for BS Information Technology (information for students) according to university policy?,Admission opening dates vary by university and intake. Please check the latest official admission schedule.
232,Admission,When does admission open for BBA (information for students) according to university policy?,Admission opening dates vary by university and intake. Please check the latest official admission schedule.
233,Admission,When does admission open for MBA (information for students) according to university policy?,Admission opening dates vary by university and intake. Please check the latest official admission schedule.
234,Admission,What documents are required for admission (information for students) online?,"Common documents include academic certificates, identification documents, photographs, and the completed application form. The university may require additional documents."
235,Admission,Can I apply online for admission (information for students) online?,Most universities provide an online admission application system. Please use the official university admission portal.
236,Admission,What is the admission deadline (information for students) online?,The admission deadline depends on the program and intake. Please check the current official admission notice.
237,Admission,Can I apply after the deadline (information for students) online?,Late applications are accepted only if the university announces an extension. Check the latest official notice.
238,Admission,How can I check my admission application status (information for students) online?,Check your application status through the university admission portal or contact the admissions office.
239,Admission,What is the admission fee (information for students) online?,The admission or application fee varies by university and program. Check the current fee schedule.
240,Admission,Can I change my selected program after applying (information for students) through the student portal?,Program changes may be possible before a specified deadline. Contact the admissions office for the current procedure.
241,Programs,What programs does the university offer (information for students) through the student portal?,The university offers programs listed in its current academic prospectus. Check the official programs and departments list.
242,Programs,Does the university offer BS Computer Science (information for students) through the student portal?,Please check the university's current programs list to confirm whether BS Computer Science is offered.
243,Programs,Does the university offer BS Artificial Intelligence (information for students) through the student portal?,Please check the university's current programs list to confirm whether BS Artificial Intelligence is offered.
244,Programs,Does the university offer BS Software Engineering (information for students) through the student portal?,Please check the university's current programs list to confirm whether BS Software Engineering is offered.
245,Programs,Does the university offer BS Information Technology (information for students) through the student portal?,Please check the university's current programs list to confirm whether BS Information Technology is offered.
246,Programs,Does the university offer BBA (information for students) at the university?,Please check the university's current programs list to confirm whether BBA is offered.
247,Programs,Does the university offer MBA (information for students) at the university?,Please check the university's current programs list to confirm whether MBA is offered.
248,Programs,How long is the BS Computer Science program (information for students) at the university?,The duration of BS Computer Science depends on the university's approved curriculum and academic regulations.
249,Programs,How long is the BS Artificial Intelligence program (information for students) at the university?,The duration of BS Artificial Intelligence depends on the university's approved curriculum and academic regulations.
250,Programs,How long is the BS Software Engineering program (information for students) at the university?,The duration of BS Software Engineering depends on the university's approved curriculum and academic regulations.
251,Programs,How long is the BS Information Technology program (information for students) at the university?,The duration of BS Information Technology depends on the university's approved curriculum and academic regulations.
252,Programs,How long is the BBA program (information for students) for current students?,The duration of BBA depends on the university's approved curriculum and academic regulations.
253,Programs,How long is the MBA program (information for students) for current students?,The duration of MBA depends on the university's approved curriculum and academic regulations.
254,Programs,What subjects are included in BS Computer Science (information for students) for current students?,The subjects for BS Computer Science are listed in the official curriculum or degree plan.
255,Programs,What subjects are included in BS Artificial Intelligence (information for students) for current students?,The subjects for BS Artificial Intelligence are listed in the official curriculum or degree plan.
256,Programs,What subjects are included in BS Software Engineering (information for students) for current students?,The subjects for BS Software Engineering are listed in the official curriculum or degree plan.
257,Programs,What subjects are included in BS Information Technology (information for students) for current students?,The subjects for BS Information Technology are listed in the official curriculum or degree plan.
258,Programs,What subjects are included in BBA (information for students) for the current academic year?,The subjects for BBA are listed in the official curriculum or degree plan.
259,Programs,What subjects are included in MBA (information for students) for the current academic year?,The subjects for MBA are listed in the official curriculum or degree plan.
260,Programs,Is BS Computer Science available in morning classes (information for students) for the current academic year?,Class timing depends on the university and department schedule. Check the current timetable.
261,Programs,Is BS Artificial Intelligence available in morning classes (information for students) for the current academic year?,Class timing depends on the university and department schedule. Check the current timetable.
262,Programs,Is BS Software Engineering available in morning classes (information for students) for the current academic year?,Class timing depends on the university and department schedule. Check the current timetable.
263,Programs,Is BS Information Technology available in morning classes (information for students) for the current academic year?,Class timing depends on the university and department schedule. Check the current timetable.
264,Programs,Is BBA available in morning classes (information for students) according to university policy?,Class timing depends on the university and department schedule. Check the current timetable.
265,Programs,Is MBA available in morning classes (information for students) according to university policy?,Class timing depends on the university and department schedule. Check the current timetable.
266,Programs,Is BS Computer Science available in evening classes (information for students) according to university policy?,Evening availability depends on the department's current schedule.
267,Programs,Is BS Artificial Intelligence available in evening classes (information for students) according to university policy?,Evening availability depends on the department's current schedule.
268,Programs,Is BS Software Engineering available in evening classes (information for students) according to university policy?,Evening availability depends on the department's current schedule.
269,Programs,Is BS Information Technology available in evening classes (information for students) according to university policy?,Evening availability depends on the department's current schedule.
270,Programs,Is BBA available in evening classes (information for students) online?,Evening availability depends on the department's current schedule.
271,Programs,Is MBA available in evening classes (information for students) online?,Evening availability depends on the department's current schedule.
272,Programs,What is the scope of BS Computer Science (information for students) online?,"The scope of BS Computer Science depends on industry demand, skills, specialization, and career opportunities."
273,Programs,Could you explain the scope of BS Artificial Intelligence online?,"The scope of BS Artificial Intelligence depends on industry demand, skills, specialization, and career opportunities."
274,Programs,What is the scope of BS Software Engineering (information for students) online?,"The scope of BS Software Engineering depends on industry demand, skills, specialization, and career opportunities."
275,Programs,What is the scope of BS Information Technology (information for students) online?,"The scope of BS Information Technology depends on industry demand, skills, specialization, and career opportunities."
276,Programs,What is the scope of BBA (information for students) through the student portal?,"The scope of BBA depends on industry demand, skills, specialization, and career opportunities."
277,Programs,What is the scope of MBA (information for students) through the student portal?,"The scope of MBA depends on industry demand, skills, specialization, and career opportunities."
278,Programs,What careers are available after BS Computer Science (information for students) through the student portal?,Career options after BS Computer Science depend on the skills and specialization you develop. Consult the program's official career information.
279,Programs,What careers are available after BS Artificial Intelligence (information for students) through the student portal?,Career options after BS Artificial Intelligence depend on the skills and specialization you develop. Consult the program's official career information.
280,Programs,What careers are available after BS Software Engineering (information for students) through the student portal?,Career options after BS Software Engineering depend on the skills and specialization you develop. Consult the program's official career information.
281,Programs,What careers are available after BS Information Technology (information for students) through the student portal?,Career options after BS Information Technology depend on the skills and specialization you develop. Consult the program's official career information.
282,Programs,What careers are available after BBA (information for students) at the university?,Career options after BBA depend on the skills and specialization you develop. Consult the program's official career information.
283,Programs,What careers are available after MBA (information for students) at the university?,Career options after MBA depend on the skills and specialization you develop. Consult the program's official career information.
284,Programs,Can I change my major (information for students) at the university?,Major or program changes are subject to university academic regulations and available seats.
285,Programs,Where can I find the degree curriculum (information for students) at the university?,"The official curriculum can usually be found on the university website, department office, or academic portal."
286,Fees,What is the tuition fee for BS Computer Science (information for students) at the university?,The tuition fee for BS Computer Science should be confirmed from the university's latest official fee structure.
287,Fees,What is the tuition fee for BS Artificial Intelligence (information for students) at the university?,The tuition fee for BS Artificial Intelligence should be confirmed from the university's latest official fee structure.
288,Fees,What is the tuition fee for BS Software Engineering (information for students) for current students?,The tuition fee for BS Software Engineering should be confirmed from the university's latest official fee structure.
289,Fees,What is the tuition fee for BS Information Technology (information for students) for current students?,The tuition fee for BS Information Technology should be confirmed from the university's latest official fee structure.
290,Fees,What is the tuition fee for BBA (information for students) for current students?,The tuition fee for BBA should be confirmed from the university's latest official fee structure.
291,Fees,Could you explain the tuition fee for MBA for current students?,The tuition fee for MBA should be confirmed from the university's latest official fee structure.
292,Fees,How can I pay my semester fee (information for students) for current students?,"Use the payment methods authorized by the university, such as its online portal or designated bank/payment channels."
293,Fees,What is the last date for fee payment (information for students) for current students?,The fee payment deadline is published in the academic calendar or fee notice.
294,Fees,Is there a late fee for delayed payment (information for students) for the current academic year?,A late payment charge may apply according to university regulations. Check the latest fee policy.
295,Fees,Can I pay my fee in installments (information for students) for the current academic year?,Installment options depend on university policy and may require approval from the relevant office.
296,Fees,How can I get my fee voucher (information for students) for the current academic year?,Fee vouchers are normally available through the student portal or accounts office.
297,Fees,What should I do if my fee payment is not updated (information for students) for the current academic year?,Keep your payment receipt and contact the accounts or finance office to have the transaction verified.
298,Fees,Can I get a fee refund (information for students) for the current academic year?,Refund eligibility depends on the university's refund policy and the reason for withdrawal or cancellation.
299,Fees,How can I request a fee refund (information for students) for the current academic year?,Submit a refund request through the procedure specified by the university finance or accounts office.
300,Fees,Where can I find the latest fee structure (information for students) according to university policy?,The latest fee structure should be obtained from the official university website or accounts office.
301,Exams,When will the midterm exams start (information for students) according to university policy?,Midterm dates are published in the university's academic calendar or examination schedule.
302,Exams,When will the final exams start (information for students) according to university policy?,Final examination dates are announced by the examination office and academic calendar.
303,Exams,Where can I find the exam timetable (information for students) according to university policy?,"Check the student portal, examination office notice board, or official university announcements."
304,Exams,What should I bring to the exam (information for students) according to university policy?,Bring the identification and stationery permitted by the university examination rules.
305,Exams,What happens if I miss an exam (information for students) according to university policy?,"If you miss an exam, follow the university's rules for absence and contact the examination office promptly."
306,Exams,Can I apply for a re-examination (information for students) online?,Re-examination opportunities depend on the university's examination regulations.
307,Exams,What is the procedure to apply for a supplementary exam online?,Apply through the examination office according to the current supplementary examination procedure.
308,Exams,What are the examination rules (information for students) online?,Examination rules are defined by the university examination regulations and should be followed by all students.
309,Exams,When are exam results announced (information for students) online?,Results are announced after evaluation and approval according to the university's examination process.
310,Exams,How can I check my exam result (information for students) online?,Check your result through the official student portal or examination office.
311,Attendance,What is the minimum attendance requirement (information for students) online?,The minimum attendance requirement is defined by the university's academic regulations. Check the current attendance policy.
312,Attendance,How can I check my attendance (information for students) through the student portal?,Attendance can usually be viewed through the student portal or confirmed by the relevant department.
313,Attendance,What happens if my attendance is below the required percentage (information for students) through the student portal?,Students below the required attendance level may face academic restrictions according to university policy.
314,Attendance,Can I request attendance correction (information for students) through the student portal?,"If your attendance record is incorrect, contact the course instructor or department office with supporting evidence."
315,Attendance,Does medical leave affect attendance (information for students) through the student portal?,Medical leave is handled according to university leave and attendance regulations.
316,Attendance,How do I apply for academic leave (information for students) through the student portal?,Submit a leave application through the procedure specified by your department or student affairs office.
317,Attendance,Can attendance be marked late (information for students) through the student portal?,Late attendance rules depend on the instructor and university attendance policy.
318,Attendance,Who maintains the attendance record (information for students) at the university?,Attendance is normally recorded by course instructors and maintained through the department or academic system.
319,Attendance,Can I attend another section to make up attendance (information for students) at the university?,Section changes or make-up attendance are allowed only when approved under university policy.
320,Attendance,Why is my attendance missing from the portal (information for students) at the university?,Contact the course instructor or department if attendance has not been updated or appears incorrect.
321,Scholarships,What scholarships are available for students (information for students) at the university?,"Available scholarships depend on the university, government programs, donors, and current scholarship announcements."
322,Scholarships,How can I apply for a scholarship (information for students) at the university?,Follow the application procedure published by the university scholarship or financial aid office.
323,Scholarships,What are the scholarship eligibility criteria (information for students) at the university?,"Eligibility varies by scholarship and may consider academic performance, financial need, or other criteria."
324,Scholarships,When is the scholarship application deadline (information for students) for current students?,Check the current scholarship announcement for the exact deadline.
325,Scholarships,Is there a merit scholarship (information for students) for current students?,"Many universities offer merit-based scholarships, but availability and criteria should be confirmed from the current official notice."
326,Scholarships,Is financial assistance available (information for students) for current students?,"Financial assistance may be available through scholarships, aid programs, or fee support. Contact the financial aid office."
327,Scholarships,Can international students apply for scholarships (information for students) for current students?,Eligibility depends on the scholarship rules and university policy.
328,Scholarships,Can I receive more than one scholarship (information for students) for current students?,Receiving multiple scholarships depends on the terms and conditions of the specific programs.
329,Scholarships,How will I know if my scholarship is approved (information for students) for current students?,"Scholarship decisions are normally communicated through the student portal, official notice, or scholarship office."
330,Scholarships,Who should I contact about scholarships (information for students) for the current academic year?,Contact the university scholarship or financial aid office for current information.
331,Library,What are the library opening hours (information for students) for the current academic year?,Library hours vary by campus and semester. Check the latest library schedule.
332,Library,How can I get a library card (information for students) for the current academic year?,Library membership is normally activated through the university library according to its registration procedure.
333,Library,How many books can I borrow (information for students) for the current academic year?,The borrowing limit depends on student status and library policy.
334,Library,How long can I keep a borrowed book (information for students) for the current academic year?,The loan period is determined by the library's current borrowing policy.
335,Library,Am I allowed to renew a borrowed book for the current academic year?,Books may be renewable if they are eligible and not reserved by another user.
336,Library,What happens if I return a book late (information for students) according to university policy?,Late returns may result in fines or borrowing restrictions according to library policy.
337,Library,What is the procedure to search for a book according to university policy?,Use the university library catalog or ask library staff for assistance.
338,Library,Can I access online journals (information for students) according to university policy?,"If the university subscribes to online databases, students can access them through the library's authorized services."
339,Library,Can I reserve a book (information for students) according to university policy?,Book reservation availability depends on the library system and current policy.
340,Library,Where is the university library (information for students) according to university policy?,The library location depends on the campus. Check the official campus map or library information.
341,Hostel,How can I apply for university hostel accommodation (information for students) according to university policy?,Submit a hostel application through the university's approved hostel application procedure.
342,Hostel,Who is eligible for hostel accommodation (information for students) online?,"Eligibility depends on university hostel rules, available rooms, and student status."
343,Hostel,What is the hostel fee (information for students) online?,Hostel fees vary by campus and room type. Check the latest official hostel fee schedule.
344,Hostel,When is hostel admission open (information for students) online?,Hostel application dates are announced by the university hostel administration.
345,Hostel,Can I choose my roommate (information for students) online?,Roommate selection depends on hostel policy and room availability.
346,Hostel,What documents are required for hostel admission (information for students) online?,"Required documents may include student identification, admission proof, photographs, and other documents specified by hostel administration."
347,Hostel,What are the hostel rules (information for students) online?,"Hostel rules cover residence, discipline, visitors, safety, and other matters. Follow the current official hostel regulations."
348,Hostel,Can I leave the hostel during the semester (information for students) through the student portal?,Hostel withdrawal or room cancellation follows the hostel administration's procedure.
349,Hostel,What is the procedure to report a hostel problem through the student portal?,"Report hostel maintenance, safety, or administrative issues to the hostel warden or designated office."
350,Hostel,Are meals provided in the hostel (information for students) through the student portal?,Meal facilities depend on the hostel and campus arrangements. Check the current hostel information.
351,Student Portal,How do I log in to the student portal (information for students) through the student portal?,Use the credentials provided by the university and access the official student portal.
352,Student Portal,I forgot my student portal password. What should I do (information for students) through the student portal?,Use the portal's password recovery option or contact the university IT/help desk.
353,Student Portal,How can I change my portal password (information for students) through the student portal?,Use the account settings or password-change option in the official student portal.
354,Student Portal,How can I update my profile information (information for students) at the university?,"Update profile information through the student portal if editing is enabled, or contact the relevant office."
355,Student Portal,Why can't I log in to the student portal (information for students) at the university?,"Check your credentials and internet connection. If the problem continues, contact the university IT support office."
356,Student Portal,How can I register for courses online (information for students) at the university?,Use the course registration feature in the student portal during the announced registration period.
357,Student Portal,How can I download my fee voucher (information for students) at the university?,Log in to the student portal and use the fee or finance section if the university provides electronic vouchers.
358,Student Portal,How can I download my transcript (information for students) at the university?,"If electronic transcripts are supported, request or download them through the student portal; otherwise contact the registrar."
359,Student Portal,How can I see my registered courses (information for students) at the university?,Open the registered courses or enrollment section of the student portal.
360,Student Portal,How can I contact technical support for the portal (information for students) for current students?,Contact the university IT help desk through the official support channel.
361,Courses,How do I register for a course (information for students) for current students?,Course registration is completed through the university's registration system during the announced registration period.
362,Courses,Can I drop a course (information for students) for current students?,Course withdrawal is allowed only within the period and conditions defined by academic regulations.
363,Courses,Can I add a course after registration (information for students) for current students?,Adding a course after registration depends on the add/drop deadline and departmental approval.
364,Courses,What is a prerequisite course (information for students) for current students?,A prerequisite is a course that must normally be completed before taking another course.
365,Courses,How can I find my course prerequisites (information for students) for current students?,Check the official degree plan or course catalog for prerequisite requirements.
366,Courses,Can I repeat a failed course (information for students) for the current academic year?,Course repetition is generally governed by the university's academic regulations.
367,Courses,What is a credit hour (information for students) for the current academic year?,A credit hour is a unit used to measure the academic workload of a course.
368,Courses,How many courses can I take in a semester (information for students) for the current academic year?,"The allowed course load depends on the program, semester, and university academic regulations."
369,Courses,Can I take an extra course (information for students) for the current academic year?,An extra course may require academic approval and must comply with the maximum credit-hour policy.
370,Courses,What happens if I withdraw from a course (information for students) for the current academic year?,The academic and transcript consequences depend on the university's withdrawal policy.
371,Faculty,How can I contact my course instructor (information for students) for the current academic year?,"Use the official university email, learning platform, department office, or other approved communication channel."
372,Faculty,Where can I find faculty office hours (information for students) according to university policy?,"Faculty office hours may be listed on the department website, student portal, or department notice board."
373,Faculty,Who is the head of my department (information for students) according to university policy?,The current department head should be confirmed from the official university department page.
374,Faculty,How can I meet a faculty member (information for students) according to university policy?,Contact the faculty member through the official channel and request an appointment during available hours.
375,Faculty,How can I submit an assignment to my instructor (information for students) according to university policy?,Submit assignments using the platform or method specified by the instructor.
376,Faculty,What should I do if I have an academic issue with a course (information for students) according to university policy?,"First discuss the issue with the course instructor, then follow the department's academic complaint procedure if necessary."
377,Faculty,How can I request a recommendation letter (information for students) according to university policy?,Ask the faculty member according to the university's recommendation or reference procedure and provide the required information.
378,Faculty,Can I change my academic advisor (information for students) online?,Advisor changes depend on departmental policy and approval.
379,Faculty,Who is my academic advisor (information for students) online?,Your academic advisor can be identified through the student portal or department office.
380,Faculty,How can I contact the department office (information for students) online?,Use the department's official contact information listed on the university website.
381,Campus Facilities,Does the university have a computer lab (information for students) online?,Check the campus facilities list to confirm available computer labs and their locations.
382,Campus Facilities,Does the university provide Wi-Fi (information for students) online?,University Wi-Fi availability and access rules depend on the campus IT policy.
383,Campus Facilities,How can I access campus Wi-Fi (information for students) online?,Use the credentials and connection instructions provided by the university IT department.
384,Campus Facilities,Is there a cafeteria on campus (information for students) through the student portal?,Cafeteria availability depends on the campus. Check the official campus facilities information.
385,Campus Facilities,Does the campus have a sports facility (information for students) through the student portal?,Sports facilities vary by campus and can be confirmed through student affairs or campus facilities information.
386,Campus Facilities,Is there a medical center on campus (information for students) through the student portal?,Check the campus facilities or student services information for available medical services.
387,Campus Facilities,Where can I report a maintenance issue (information for students) through the student portal?,Report maintenance problems to the campus facilities or administration office through the approved channel.
388,Campus Facilities,Is parking available for students (information for students) through the student portal?,Student parking availability and permits depend on campus parking rules.
389,Campus Facilities,How can I get a student ID card (information for students) through the student portal?,Student ID cards are issued through the university's designated student services or administration office.
390,Campus Facilities,What should I do if I lose my student ID card (information for students) at the university?,Report the lost card immediately to student services and follow the replacement procedure.
391,Graduation,What are the graduation requirements (information for students) at the university?,Graduation requirements are defined by the degree plan and university academic regulations.
392,Graduation,How can I apply for graduation (information for students) at the university?,Submit a graduation application through the registrar or student portal according to the announced procedure.
393,Graduation,When should I apply for graduation (information for students) at the university?,Apply during the graduation application period announced by the registrar.
394,Graduation,How can I check whether I have completed my degree requirements (information for students) at the university?,Review your degree audit or contact your academic advisor or registrar.
395,Graduation,What documents are needed for graduation clearance (information for students) at the university?,"Required documents vary by university and may include clearance forms, identification, and financial or departmental clearance."
396,Graduation,When is the graduation ceremony (information for students) for current students?,Ceremony dates are announced by the university through official notices.
397,Graduation,What is the procedure to get my degree certificate for current students?,Degree certificates are issued by the registrar or examination authority after completion of all requirements.
398,Graduation,Can I attend graduation if my clearance is incomplete (information for students) for current students?,Participation depends on the university's graduation and clearance rules.
399,Graduation,How can I request a duplicate degree (information for students) for current students?,Follow the registrar's procedure for replacement or duplicate degree documents.
400,Graduation,How can I obtain an official transcript after graduation (information for students) for current students?,Request an official transcript through the registrar or examination office.
401,Rules & Policies,Where can I find university rules (information for students) for current students?,"University rules are normally published in official regulations, student handbooks, or the university website."
402,Rules & Policies,What is the student code of conduct (information for students) for the current academic year?,The student code of conduct defines expected academic and disciplinary behavior.
403,Rules & Policies,What happens if a student violates university rules (information for students) for the current academic year?,Disciplinary action depends on the nature of the violation and the university's disciplinary regulations.
404,Rules & Policies,How can I submit a complaint (information for students) for the current academic year?,Use the official complaint or grievance procedure provided by student affairs or the relevant office.
405,Rules & Policies,How can I appeal an academic decision (information for students) for the current academic year?,Follow the university's formal academic appeal procedure within the specified deadline.
406,Rules & Policies,What is the plagiarism policy (information for students) for the current academic year?,Students must follow the university's academic integrity and plagiarism policy.
407,Rules & Policies,What is considered academic misconduct (information for students) for the current academic year?,"Academic misconduct may include plagiarism, cheating, unauthorized collaboration, or other violations defined by university regulations."
408,Rules & Policies,Can students use mobile phones during exams (information for students) according to university policy?,Mobile phone use during exams is governed by examination rules and may be prohibited.
409,Rules & Policies,What is the dress code (information for students) according to university policy?,"Dress requirements, if any, are defined by the university or campus policy."
410,Rules & Policies,Where can I report a disciplinary concern (information for students) according to university policy?,"Report concerns through the university's designated student affairs, discipline, or complaint channel."
411,Timetable,Where can I find my class timetable (information for students) according to university policy?,"Check the student portal, department notice board, or official timetable announcement."
412,Timetable,How can I know my classroom (information for students) according to university policy?,Classroom assignments are usually shown in the timetable or announced by the department.
413,Timetable,Can the timetable change during the semester (information for students) according to university policy?,"Yes, timetable changes may occur due to academic or administrative requirements. Check official announcements."
414,Timetable,How can I report a timetable conflict (information for students) online?,Report schedule conflicts to the department or academic office promptly.
415,Timetable,What is the procedure to find the timetable for my department online?,Check the department's official timetable announcement or student portal.
416,Timetable,What should I do if two classes are scheduled at the same time (information for students) online?,Contact the department or academic office to report the conflict and request guidance.
417,Timetable,Where can I find room numbers (information for students) online?,Room numbers are normally listed with course schedules or posted by the department.
418,Timetable,How do I know if a class has been cancelled (information for students) online?,"Check official university announcements, the student portal, or communication from the instructor."
419,Timetable,Am I allowed to change my class section online?,Section changes depend on available seats and department approval.
420,Timetable,When is the class schedule published (information for students) through the student portal?,The schedule is normally published before the start of the academic term.
421,General Student Services,What is the procedure to contact student affairs through the student portal?,Use the official student affairs office contact details published by the university.
422,General Student Services,Where is the registrar office (information for students) through the student portal?,The registrar office location depends on the campus. Check the official campus directory.
423,General Student Services,What does the registrar office handle (information for students) through the student portal?,"The registrar commonly handles academic records, enrollment, transcripts, and graduation documentation."
424,General Student Services,How can I request an enrollment certificate (information for students) through the student portal?,Request an enrollment certificate through the registrar or student portal according to university procedure.
425,General Student Services,How can I request a bonafide student certificate (information for students) through the student portal?,Apply through the registrar or designated student services office.
426,General Student Services,How can I change my personal information (information for students) at the university?,Submit an update request through the student portal or registrar with required supporting documents.
427,General Student Services,What is the procedure to request a transcript at the university?,Submit a transcript request through the registrar or examination office.
428,General Student Services,How can I get an official university letter (information for students) at the university?,Request the required letter from the relevant administrative office using the university's official procedure.
429,General Student Services,Could you explain the university contact number at the university?,Use the current official contact number published on the university website.
430,General Student Services,What is the university email address (information for students) at the university?,Use the current official email address published by the university.
431,Admission,What are the admission requirements for BS Computer Science (information for students) at the university?,Admission requirements for BS Computer Science depend on the university's current eligibility policy. Please check the official admissions requirements.
432,Admission,How can I apply for admission to BS Computer Science (information for students) for current students?,You can apply for BS Computer Science through the university's official admission portal or admissions office.
433,Admission,How can I apply for admission to BS Artificial Intelligence (information for students) for the current academic year?,You can apply for BS Artificial Intelligence through the university's official admission portal or admissions office.
434,Admission,What is the procedure to apply for admission to BS Software Engineering for the current academic year?,You can apply for BS Software Engineering through the university's official admission portal or admissions office.
435,Admission,When does admission open for BS Computer Science (information for students) for the current academic year?,Admission opening dates vary by university and intake. Please check the latest official admission schedule.
436,Admission,What documents are required for admission (information for students) according to university policy?,"Common documents include academic certificates, identification documents, photographs, and the completed application form. The university may require additional documents."
437,Admission,Am I allowed to change my selected program after applying online?,Program changes may be possible before a specified deadline. Contact the admissions office for the current procedure.
438,Programs,Does the university offer BBA (information for students) through the student portal?,Please check the university's current programs list to confirm whether BBA is offered.
439,Programs,How long is the BBA program (information for students) at the university?,The duration of BBA depends on the university's approved curriculum and academic regulations.
440,Programs,What subjects are included in BBA (information for students) for current students?,The subjects for BBA are listed in the official curriculum or degree plan.
441,Programs,Is BBA available in morning classes (information for students) for the current academic year?,Class timing depends on the university and department schedule. Check the current timetable.
442,Programs,Is BBA available in evening classes (information for students) according to university policy?,Evening availability depends on the department's current schedule.
443,Programs,What is the scope of BS Artificial Intelligence (information for students) online?,"The scope of BS Artificial Intelligence depends on industry demand, skills, specialization, and career opportunities."
444,Programs,Could you explain the scope of BS Software Engineering online?,"The scope of BS Software Engineering depends on industry demand, skills, specialization, and career opportunities."
445,Programs,What is the scope of BBA (information for students) online?,"The scope of BBA depends on industry demand, skills, specialization, and career opportunities."
446,Programs,What careers are available after BBA (information for students) through the student portal?,Career options after BBA depend on the skills and specialization you develop. Consult the program's official career information.
447,Programs,Where do I check the degree curriculum at the university?,"The official curriculum can usually be found on the university website, department office, or academic portal."
448,Fees,Could you explain the tuition fee for BS Computer Science at the university?,The tuition fee for BS Computer Science should be confirmed from the university's latest official fee structure.
449,Fees,What is the tuition fee for BS Software Engineering (information for students) at the university?,The tuition fee for BS Software Engineering should be confirmed from the university's latest official fee structure.
450,Fees,What is the tuition fee for MBA (information for students) for current students?,The tuition fee for MBA should be confirmed from the university's latest official fee structure.
451,Fees,Is there a late fee for delayed payment (information for students) for current students?,A late payment charge may apply according to university regulations. Check the latest fee policy.
452,Fees,What is the procedure to get my fee voucher for the current academic year?,Fee vouchers are normally available through the student portal or accounts office.
453,Fees,Where can I find the latest fee structure (information for students) for the current academic year?,The latest fee structure should be obtained from the official university website or accounts office.
454,Exams,Where do I check the exam timetable according to university policy?,"Check the student portal, examination office notice board, or official university announcements."
455,Exams,Am I allowed to apply for a re-examination according to university policy?,Re-examination opportunities depend on the university's examination regulations.
456,Exams,How can I apply for a supplementary exam (information for students) online?,Apply through the examination office according to the current supplementary examination procedure.
457,Attendance,How can I check my attendance (information for students) online?,Attendance can usually be viewed through the student portal or confirmed by the relevant department.
458,Attendance,Who maintains the attendance record (information for students) through the student portal?,Attendance is normally recorded by course instructors and maintained through the department or academic system.
459,Scholarships,When is the scholarship application deadline (information for students) at the university?,Check the current scholarship announcement for the exact deadline.
460,Scholarships,Who should I contact about scholarships (information for students) for current students?,Contact the university scholarship or financial aid office for current information.
461,Library,What is the procedure to get a library card for the current academic year?,Library membership is normally activated through the university library according to its registration procedure.
462,Library,Can I renew a borrowed book (information for students) for the current academic year?,Books may be renewable if they are eligible and not reserved by another user.
463,Library,What happens if I return a book late (information for students) for the current academic year?,Late returns may result in fines or borrowing restrictions according to library policy.
464,Library,How can I search for a book (information for students) according to university policy?,Use the university library catalog or ask library staff for assistance.
465,Hostel,Who is eligible for hostel accommodation (information for students) according to university policy?,"Eligibility depends on university hostel rules, available rooms, and student status."
466,Hostel,Am I allowed to leave the hostel during the semester online?,Hostel withdrawal or room cancellation follows the hostel administration's procedure.
467,Hostel,How can I report a hostel problem (information for students) through the student portal?,"Report hostel maintenance, safety, or administrative issues to the hostel warden or designated office."
468,Student Portal,How can I update my profile information (information for students) through the student portal?,"Update profile information through the student portal if editing is enabled, or contact the relevant office."
469,Student Portal,What is the procedure to register for courses online at the university?,Use the course registration feature in the student portal during the announced registration period.
470,Student Portal,How can I contact technical support for the portal (information for students) at the university?,Contact the university IT help desk through the official support channel.
471,Courses,Could you explain a prerequisite course for current students?,A prerequisite is a course that must normally be completed before taking another course.
472,Courses,Am I allowed to repeat a failed course for current students?,Course repetition is generally governed by the university's academic regulations.
473,Faculty,Where can I find faculty office hours (information for students) for the current academic year?,"Faculty office hours may be listed on the department website, student portal, or department notice board."
474,Faculty,What is the procedure to meet a faculty member according to university policy?,Contact the faculty member through the official channel and request an appointment during available hours.
475,Faculty,Am I allowed to change my academic advisor according to university policy?,Advisor changes depend on departmental policy and approval.
476,Faculty,What is the procedure to contact the department office online?,Use the department's official contact information listed on the university website.
477,Campus Facilities,Is there a cafeteria on campus (information for students) online?,Cafeteria availability depends on the campus. Check the official campus facilities information.
478,Campus Facilities,What should I do if I lose my student ID card (information for students) through the student portal?,Report the lost card immediately to student services and follow the replacement procedure.
479,Graduation,What is the procedure to apply for graduation at the university?,Submit a graduation application through the registrar or student portal according to the announced procedure.
480,Graduation,When is the graduation ceremony (information for students) at the university?,Ceremony dates are announced by the university through official notices.
481,Graduation,How can I get my degree certificate (information for students) for current students?,Degree certificates are issued by the registrar or examination authority after completion of all requirements.
482,Rules & Policies,What is the student code of conduct (information for students) for current students?,The student code of conduct defines expected academic and disciplinary behavior.
483,Rules & Policies,What is the procedure to submit a complaint for the current academic year?,Use the official complaint or grievance procedure provided by student affairs or the relevant office.
484,Rules & Policies,Could you explain the plagiarism policy for the current academic year?,Students must follow the university's academic integrity and plagiarism policy.
485,Rules & Policies,Can students use mobile phones during exams (information for students) for the current academic year?,Mobile phone use during exams is governed by examination rules and may be prohibited.
486,Timetable,Where do I check my class timetable according to university policy?,"Check the student portal, department notice board, or official timetable announcement."
487,Timetable,How can I report a timetable conflict (information for students) according to university policy?,Report schedule conflicts to the department or academic office promptly.
488,Timetable,How can I find the timetable for my department (information for students) online?,Check the department's official timetable announcement or student portal.
489,Timetable,Where do I check room numbers online?,Room numbers are normally listed with course schedules or posted by the department.
490,Timetable,Can I change my class section (information for students) online?,Section changes depend on available seats and department approval.
491,Timetable,When is the class schedule published (information for students) online?,The schedule is normally published before the start of the academic term.
492,General Student Services,How can I contact student affairs (information for students) through the student portal?,Use the official student affairs office contact details published by the university.
493,General Student Services,How can I change my personal information (information for students) through the student portal?,Submit an update request through the student portal or registrar with required supporting documents.
494,General Student Services,How can I request a transcript (information for students) at the university?,Submit a transcript request through the registrar or examination office.
495,General Student Services,What is the procedure to get an official university letter at the university?,Request the required letter from the relevant administrative office using the university's official procedure.
496,General Student Services,What is the university contact number (information for students) at the university?,Use the current official contact number published on the university website.
497,General Student Services,Could you explain the university email address at the university?,Use the current official email address published by the university.
498,Admission,What are the admission requirements for BS Artificial Intelligence (information for students) at the university?,Admission requirements for BS Artificial Intelligence depend on the university's current eligibility policy. Please check the official admissions requirements.
499,Admission,How can I apply for admission to BS Artificial Intelligence (information for students) for current students?,You can apply for BS Artificial Intelligence through the university's official admission portal or admissions office.
500,Admission,What is the procedure to apply for admission to BS Information Technology for the current academic year?,You can apply for BS Information Technology through the university's official admission portal or admissions office.
501,Admission,When does admission open for BS Artificial Intelligence (information for students) for the current academic year?,Admission opening dates vary by university and intake. Please check the latest official admission schedule.
502,Admission,Am I allowed to apply online for admission according to university policy?,Most universities provide an online admission application system. Please use the official university admission portal.
503,Admission,Could you explain the admission fee online?,The admission or application fee varies by university and program. Check the current fee schedule.
504,Admission,Can I change my selected program after applying (information for students) online?,Program changes may be possible before a specified deadline. Contact the admissions office for the current procedure.
505,Programs,What programs does the university offer (information for students) online?,The university offers programs listed in its current academic prospectus. Check the official programs and departments list.
506,Programs,Does the university offer MBA (information for students) through the student portal?,Please check the university's current programs list to confirm whether MBA is offered.
507,Programs,How long is the MBA program (information for students) at the university?,The duration of MBA depends on the university's approved curriculum and academic regulations.
508,Programs,What subjects are included in MBA (information for students) for current students?,The subjects for MBA are listed in the official curriculum or degree plan.
509,Programs,Is MBA available in morning classes (information for students) for the current academic year?,Class timing depends on the university and department schedule. Check the current timetable.
510,Programs,Is MBA available in evening classes (information for students) according to university policy?,Evening availability depends on the department's current schedule.
511,Programs,Could you explain the scope of BS Information Technology online?,"The scope of BS Information Technology depends on industry demand, skills, specialization, and career opportunities."
512,Programs,What is the scope of MBA (information for students) online?,"The scope of MBA depends on industry demand, skills, specialization, and career opportunities."
513,Programs,What careers are available after MBA (information for students) through the student portal?,Career options after MBA depend on the skills and specialization you develop. Consult the program's official career information.
514,Fees,Could you explain the tuition fee for BS Artificial Intelligence at the university?,The tuition fee for BS Artificial Intelligence should be confirmed from the university's latest official fee structure.
515,Fees,What is the tuition fee for BS Information Technology (information for students) at the university?,The tuition fee for BS Information Technology should be confirmed from the university's latest official fee structure.
516,Fees,Could you explain the last date for fee payment for current students?,The fee payment deadline is published in the academic calendar or fee notice.
517,Fees,Am I allowed to pay my fee in installments for current students?,Installment options depend on university policy and may require approval from the relevant office.
518,Exams,When will the midterm exams start (information for students) for the current academic year?,Midterm dates are published in the university's academic calendar or examination schedule.
519,Exams,Can I apply for a re-examination (information for students) according to university policy?,Re-examination opportunities depend on the university's examination regulations.
520,Exams,How can I apply for a supplementary exam (information for students) according to university policy?,Apply through the examination office according to the current supplementary examination procedure.
521,Attendance,Could you explain the minimum attendance requirement online?,The minimum attendance requirement is defined by the university's academic regulations. Check the current attendance policy.
522,Attendance,What happens if my attendance is below the required percentage (information for students) online?,Students below the required attendance level may face academic restrictions according to university policy.
523,Attendance,Am I allowed to attend another section to make up attendance through the student portal?,Section changes or make-up attendance are allowed only when approved under university policy.
524,Scholarships,Is there a merit scholarship (information for students) at the university?,"Many universities offer merit-based scholarships, but availability and criteria should be confirmed from the current official notice."
525,Library,What are the library opening hours (information for students) for current students?,Library hours vary by campus and semester. Check the latest library schedule.
526,Library,How can I search for a book (information for students) for the current academic year?,Use the university library catalog or ask library staff for assistance.
527,Hostel,What is the hostel fee (information for students) according to university policy?,Hostel fees vary by campus and room type. Check the latest official hostel fee schedule.
528,Hostel,Can I leave the hostel during the semester (information for students) online?,Hostel withdrawal or room cancellation follows the hostel administration's procedure.
529,Hostel,How can I report a hostel problem (information for students) online?,"Report hostel maintenance, safety, or administrative issues to the hostel warden or designated office."
530,Student Portal,Why can't I log in to the student portal (information for students) through the student portal?,"Check your credentials and internet connection. If the problem continues, contact the university IT support office."
531,Student Portal,What is the procedure to download my fee voucher at the university?,Log in to the student portal and use the fee or finance section if the university provides electronic vouchers.
532,Courses,How do I register for a course (information for students) at the university?,Course registration is completed through the university's registration system during the announced registration period.
533,Courses,Can I repeat a failed course (information for students) for current students?,Course repetition is generally governed by the university's academic regulations.
534,Courses,What is a credit hour (information for students) for current students?,A credit hour is a unit used to measure the academic workload of a course.
535,Faculty,Who is the head of my department (information for students) for the current academic year?,The current department head should be confirmed from the official university department page.
536,Faculty,What is the procedure to submit an assignment to my instructor according to university policy?,Submit assignments using the platform or method specified by the instructor.
537,Faculty,Can I change my academic advisor (information for students) according to university policy?,Advisor changes depend on departmental policy and approval.
538,Faculty,Who is my academic advisor (information for students) according to university policy?,Your academic advisor can be identified through the student portal or department office.
539,Campus Facilities,Does the campus have a sports facility (information for students) online?,Sports facilities vary by campus and can be confirmed through student affairs or campus facilities information.
540,Graduation,What are the graduation requirements (information for students) through the student portal?,Graduation requirements are defined by the degree plan and university academic regulations.
541,Graduation,How can I get my degree certificate (information for students) at the university?,Degree certificates are issued by the registrar or examination authority after completion of all requirements.
542,Graduation,What is the procedure to request a duplicate degree for current students?,Follow the registrar's procedure for replacement or duplicate degree documents.
543,Rules & Policies,What happens if a student violates university rules (information for students) for current students?,Disciplinary action depends on the nature of the violation and the university's disciplinary regulations.
544,Rules & Policies,What is the procedure to appeal an academic decision for the current academic year?,Follow the university's formal academic appeal procedure within the specified deadline.
545,Rules & Policies,Could you explain considered academic misconduct for the current academic year?,"Academic misconduct may include plagiarism, cheating, unauthorized collaboration, or other violations defined by university regulations."
546,Rules & Policies,What is the dress code (information for students) for the current academic year?,"Dress requirements, if any, are defined by the university or campus policy."
547,Timetable,How can I find the timetable for my department (information for students) according to university policy?,Check the department's official timetable announcement or student portal.
548,General Student Services,How can I contact student affairs (information for students) online?,Use the official student affairs office contact details published by the university.
549,General Student Services,How can I request a transcript (information for students) through the student portal?,Submit a transcript request through the registrar or examination office.
550,Admission,What are the admission requirements for BS Software Engineering (information for students) at the university?,Admission requirements for BS Software Engineering depend on the university's current eligibility policy. Please check the official admissions requirements.
551,Admission,How can I apply for admission to BS Software Engineering (information for students) for current students?,You can apply for BS Software Engineering through the university's official admission portal or admissions office.
552,Admission,What is the procedure to apply for admission to BBA for the current academic year?,You can apply for BBA through the university's official admission portal or admissions office.
553,Admission,When does admission open for BS Software Engineering (information for students) for the current academic year?,Admission opening dates vary by university and intake. Please check the latest official admission schedule.
554,Admission,Can I apply online for admission (information for students) according to university policy?,Most universities provide an online admission application system. Please use the official university admission portal.
555,Admission,What is the admission deadline (information for students) according to university policy?,The admission deadline depends on the program and intake. Please check the current official admission notice.
556,Admission,What is the procedure to check my admission application status online?,Check your application status through the university admission portal or contact the admissions office.
557,Programs,Does the university offer BS Computer Science (information for students) online?,Please check the university's current programs list to confirm whether BS Computer Science is offered.
558,Programs,How long is the BS Computer Science program (information for students) through the student portal?,The duration of BS Computer Science depends on the university's approved curriculum and academic regulations.
559,Programs,What subjects are included in BS Computer Science (information for students) at the university?,The subjects for BS Computer Science are listed in the official curriculum or degree plan.
560,Programs,Is BS Computer Science available in morning classes (information for students) for current students?,Class timing depends on the university and department schedule. Check the current timetable.
561,Programs,Is BS Computer Science available in evening classes (information for students) for the current academic year?,Evening availability depends on the department's current schedule.
562,Programs,What is the scope of BS Computer Science (information for students) according to university policy?,"The scope of BS Computer Science depends on industry demand, skills, specialization, and career opportunities."
563,Programs,Could you explain the scope of BBA online?,"The scope of BBA depends on industry demand, skills, specialization, and career opportunities."
564,Programs,What careers are available after BS Computer Science (information for students) online?,Career options after BS Computer Science depend on the skills and specialization you develop. Consult the program's official career information.
565,Programs,Am I allowed to change my major through the student portal?,Major or program changes are subject to university academic regulations and available seats.
566,Fees,Could you explain the tuition fee for BS Software Engineering at the university?,The tuition fee for BS Software Engineering should be confirmed from the university's latest official fee structure.
567,Fees,What is the tuition fee for BBA (information for students) at the university?,The tuition fee for BBA should be confirmed from the university's latest official fee structure.
568,Fees,What is the procedure to pay my semester fee for current students?,"Use the payment methods authorized by the university, such as its online portal or designated bank/payment channels."
569,Fees,Can I pay my fee in installments (information for students) for current students?,Installment options depend on university policy and may require approval from the relevant office.
570,Fees,How can I get my fee voucher (information for students) for current students?,Fee vouchers are normally available through the student portal or accounts office.
571,Exams,When are the midterm exams start for the current academic year?,Midterm dates are published in the university's academic calendar or examination schedule.
572,Exams,When will the final exams start (information for students) for the current academic year?,Final examination dates are announced by the examination office and academic calendar.
573,Exams,What are the examination rules (information for students) according to university policy?,Examination rules are defined by the university examination regulations and should be followed by all students.
574,Exams,What is the procedure to check my exam result online?,Check your result through the official student portal or examination office.
575,Attendance,Am I allowed to request attendance correction online?,"If your attendance record is incorrect, contact the course instructor or department office with supporting evidence."
576,Attendance,Can I attend another section to make up attendance (information for students) through the student portal?,Section changes or make-up attendance are allowed only when approved under university policy.
577,Attendance,Why is my attendance missing from the portal (information for students) through the student portal?,Contact the course instructor or department if attendance has not been updated or appears incorrect.
578,Scholarships,What is the procedure to apply for a scholarship at the university?,Follow the application procedure published by the university scholarship or financial aid office.
579,Scholarships,Is financial assistance available (information for students) at the university?,"Financial assistance may be available through scholarships, aid programs, or fee support. Contact the financial aid office."
580,Library,How can I get a library card (information for students) for current students?,Library membership is normally activated through the university library according to its registration procedure.
581,Library,Am I allowed to access online journals for the current academic year?,"If the university subscribes to online databases, students can access them through the library's authorized services."
582,Hostel,When is hostel admission open (information for students) according to university policy?,Hostel application dates are announced by the university hostel administration.
583,Hostel,Are meals provided in the hostel (information for students) online?,Meal facilities depend on the hostel and campus arrangements. Check the current hostel information.
584,Admission,Please explain what are the admission requirements for bs computer science for a university student?,Admission requirements for BS Computer Science depend on the university's current eligibility policy. Please check the official admissions requirements.
585,Admission,Please explain what are the admission requirements for bs artificial intelligence for a university student?,Admission requirements for BS Artificial Intelligence depend on the university's current eligibility policy. Please check the official admissions requirements.
586,Admission,Please explain what are the admission requirements for bs software engineering for a university student?,Admission requirements for BS Software Engineering depend on the university's current eligibility policy. Please check the official admissions requirements.
587,Admission,Please explain what are the admission requirements for bs information technology for a university student?,Admission requirements for BS Information Technology depend on the university's current eligibility policy. Please check the official admissions requirements.
588,Admission,Please explain what are the admission requirements for bba for a university student?,Admission requirements for BBA depend on the university's current eligibility policy. Please check the official admissions requirements.
589,Admission,Please explain what are the admission requirements for mba for a university student?,Admission requirements for MBA depend on the university's current eligibility policy. Please check the official admissions requirements.
590,Admission,Please explain how can i apply for admission to bs computer science for a university student?,You can apply for BS Computer Science through the university's official admission portal or admissions office.
591,Admission,Please explain how can i apply for admission to bs artificial intelligence for a university student?,You can apply for BS Artificial Intelligence through the university's official admission portal or admissions office.
592,Admission,Please explain how can i apply for admission to bs software engineering for a university student?,You can apply for BS Software Engineering through the university's official admission portal or admissions office.
593,Admission,Please explain how can i apply for admission to bs information technology for a university student?,You can apply for BS Information Technology through the university's official admission portal or admissions office.
594,Admission,Please explain how can i apply for admission to bba for a university student?,You can apply for BBA through the university's official admission portal or admissions office.
595,Admission,Please explain how can i apply for admission to mba for a university student?,You can apply for MBA through the university's official admission portal or admissions office.
596,Admission,Please explain when does admission open for bs computer science for a university student?,Admission opening dates vary by university and intake. Please check the latest official admission schedule.
597,Admission,Please explain when does admission open for bs artificial intelligence for a university student?,Admission opening dates vary by university and intake. Please check the latest official admission schedule.
598,Admission,Please explain when does admission open for bs software engineering for a university student?,Admission opening dates vary by university and intake. Please check the latest official admission schedule.
599,Admission,Please explain when does admission open for bs information technology for a university student?,Admission opening dates vary by university and intake. Please check the latest official admission schedule.
600,Admission,Please explain when does admission open for bba for a university student?,Admission opening dates vary by university and intake. Please check the latest official admission schedule.
601,Admission,Please explain when does admission open for mba for a university student?,Admission opening dates vary by university and intake. Please check the latest official admission schedule.
602,Admission,Please explain what documents are required for admission for a university student?,"Common documents include academic certificates, identification documents, photographs, and the completed application form. The university may require additional documents."
603,Admission,Please explain can i apply online for admission for a university student?,Most universities provide an online admission application system. Please use the official university admission portal.
604,Admission,Please explain what is the admission deadline for a university student?,The admission deadline depends on the program and intake. Please check the current official admission notice.
605,Admission,Please explain can i apply after the deadline for a university student?,Late applications are accepted only if the university announces an extension. Check the latest official notice.
606,Admission,Please explain how can i check my admission application status for a university student?,Check your application status through the university admission portal or contact the admissions office.
607,Admission,Please explain what is the admission fee for a university student?,The admission or application fee varies by university and program. Check the current fee schedule.
608,Admission,Please explain can i change my selected program after applying for a university student?,Program changes may be possible before a specified deadline. Contact the admissions office for the current procedure.
609,Programs,Please explain what programs does the university offer for a university student?,The university offers programs listed in its current academic prospectus. Check the official programs and departments list.
610,Programs,Please explain does the university offer bs computer science for a university student?,Please check the university's current programs list to confirm whether BS Computer Science is offered.
611,Programs,Please explain does the university offer bs artificial intelligence for a university student?,Please check the university's current programs list to confirm whether BS Artificial Intelligence is offered.
612,Programs,Please explain does the university offer bs software engineering for a university student?,Please check the university's current programs list to confirm whether BS Software Engineering is offered.
613,Programs,Please explain does the university offer bs information technology for a university student?,Please check the university's current programs list to confirm whether BS Information Technology is offered.
614,Programs,Please explain does the university offer bba for a university student?,Please check the university's current programs list to confirm whether BBA is offered.
615,Programs,Please explain does the university offer mba for a university student?,Please check the university's current programs list to confirm whether MBA is offered.
616,Programs,Please explain how long is the bs computer science program for a university student?,The duration of BS Computer Science depends on the university's approved curriculum and academic regulations.
617,Programs,Please explain how long is the bs artificial intelligence program for a university student?,The duration of BS Artificial Intelligence depends on the university's approved curriculum and academic regulations.
618,Programs,Please explain how long is the bs software engineering program for a university student?,The duration of BS Software Engineering depends on the university's approved curriculum and academic regulations.
619,Programs,Please explain how long is the bs information technology program for a university student?,The duration of BS Information Technology depends on the university's approved curriculum and academic regulations.
620,Programs,Please explain how long is the bba program for a university student?,The duration of BBA depends on the university's approved curriculum and academic regulations.
621,Programs,Please explain how long is the mba program for a university student?,The duration of MBA depends on the university's approved curriculum and academic regulations.
622,Programs,Please explain what subjects are included in bs computer science for a university student?,The subjects for BS Computer Science are listed in the official curriculum or degree plan.
623,Programs,Please explain what subjects are included in bs artificial intelligence for a university student?,The subjects for BS Artificial Intelligence are listed in the official curriculum or degree plan.
624,Programs,Please explain what subjects are included in bs software engineering for a university student?,The subjects for BS Software Engineering are listed in the official curriculum or degree plan.
625,Programs,Please explain what subjects are included in bs information technology for a university student?,The subjects for BS Information Technology are listed in the official curriculum or degree plan.
626,Programs,Please explain what subjects are included in bba for a university student?,The subjects for BBA are listed in the official curriculum or degree plan.
627,Programs,Please explain what subjects are included in mba for a university student?,The subjects for MBA are listed in the official curriculum or degree plan.
628,Programs,Please explain is bs computer science available in morning classes for a university student?,Class timing depends on the university and department schedule. Check the current timetable.
629,Programs,Please explain is bs artificial intelligence available in morning classes for a university student?,Class timing depends on the university and department schedule. Check the current timetable.
630,Programs,Please explain is bs software engineering available in morning classes for a university student?,Class timing depends on the university and department schedule. Check the current timetable.
631,Programs,Please explain is bs information technology available in morning classes for a university student?,Class timing depends on the university and department schedule. Check the current timetable.
632,Programs,Please explain is bba available in morning classes for a university student?,Class timing depends on the university and department schedule. Check the current timetable.
633,Programs,Please explain is mba available in morning classes for a university student?,Class timing depends on the university and department schedule. Check the current timetable.
634,Programs,Please explain is bs computer science available in evening classes for a university student?,Evening availability depends on the department's current schedule.
635,Programs,Please explain is bs artificial intelligence available in evening classes for a university student?,Evening availability depends on the department's current schedule.
636,Programs,Please explain is bs software engineering available in evening classes for a university student?,Evening availability depends on the department's current schedule.
637,Programs,Please explain is bs information technology available in evening classes for a university student?,Evening availability depends on the department's current schedule.
638,Programs,Please explain is bba available in evening classes for a university student?,Evening availability depends on the department's current schedule.
639,Programs,Please explain is mba available in evening classes for a university student?,Evening availability depends on the department's current schedule.
640,Programs,Please explain what is the scope of bs computer science for a university student?,"The scope of BS Computer Science depends on industry demand, skills, specialization, and career opportunities."
641,Programs,Please explain what is the scope of bs artificial intelligence for a university student?,"The scope of BS Artificial Intelligence depends on industry demand, skills, specialization, and career opportunities."
642,Programs,Please explain what is the scope of bs software engineering for a university student?,"The scope of BS Software Engineering depends on industry demand, skills, specialization, and career opportunities."
643,Programs,Please explain what is the scope of bs information technology for a university student?,"The scope of BS Information Technology depends on industry demand, skills, specialization, and career opportunities."
644,Programs,Please explain what is the scope of bba for a university student?,"The scope of BBA depends on industry demand, skills, specialization, and career opportunities."
645,Programs,Please explain what is the scope of mba for a university student?,"The scope of MBA depends on industry demand, skills, specialization, and career opportunities."
646,Programs,Please explain what careers are available after bs computer science for a university student?,Career options after BS Computer Science depend on the skills and specialization you develop. Consult the program's official career information.
647,Programs,Please explain what careers are available after bs artificial intelligence for a university student?,Career options after BS Artificial Intelligence depend on the skills and specialization you develop. Consult the program's official career information.
648,Programs,Please explain what careers are available after bs software engineering for a university student?,Career options after BS Software Engineering depend on the skills and specialization you develop. Consult the program's official career information.
649,Programs,Please explain what careers are available after bs information technology for a university student?,Career options after BS Information Technology depend on the skills and specialization you develop. Consult the program's official career information.
650,Programs,Please explain what careers are available after bba for a university student?,Career options after BBA depend on the skills and specialization you develop. Consult the program's official career information.
651,Programs,Please explain what careers are available after mba for a university student?,Career options after MBA depend on the skills and specialization you develop. Consult the program's official career information.
652,Programs,Please explain can i change my major for a university student?,Major or program changes are subject to university academic regulations and available seats.
653,Programs,Please explain where can i find the degree curriculum for a university student?,"The official curriculum can usually be found on the university website, department office, or academic portal."
654,Fees,Please explain what is the tuition fee for bs computer science for a university student?,The tuition fee for BS Computer Science should be confirmed from the university's latest official fee structure.
655,Fees,Please explain what is the tuition fee for bs artificial intelligence for a university student?,The tuition fee for BS Artificial Intelligence should be confirmed from the university's latest official fee structure.
656,Fees,Please explain what is the tuition fee for bs software engineering for a university student?,The tuition fee for BS Software Engineering should be confirmed from the university's latest official fee structure.
657,Fees,Please explain what is the tuition fee for bs information technology for a university student?,The tuition fee for BS Information Technology should be confirmed from the university's latest official fee structure.
658,Fees,Please explain what is the tuition fee for bba for a university student?,The tuition fee for BBA should be confirmed from the university's latest official fee structure.
659,Fees,Please explain what is the tuition fee for mba for a university student?,The tuition fee for MBA should be confirmed from the university's latest official fee structure.
660,Fees,Please explain how can i pay my semester fee for a university student?,"Use the payment methods authorized by the university, such as its online portal or designated bank/payment channels."
661,Fees,Please explain what is the last date for fee payment for a university student?,The fee payment deadline is published in the academic calendar or fee notice.
662,Fees,Please explain is there a late fee for delayed payment for a university student?,A late payment charge may apply according to university regulations. Check the latest fee policy.
663,Fees,Please explain can i pay my fee in installments for a university student?,Installment options depend on university policy and may require approval from the relevant office.
664,Fees,Please explain how can i get my fee voucher for a university student?,Fee vouchers are normally available through the student portal or accounts office.
665,Fees,Please explain what should i do if my fee payment is not updated for a university student?,Keep your payment receipt and contact the accounts or finance office to have the transaction verified.
666,Fees,Please explain can i get a fee refund for a university student?,Refund eligibility depends on the university's refund policy and the reason for withdrawal or cancellation.
667,Fees,Please explain how can i request a fee refund for a university student?,Submit a refund request through the procedure specified by the university finance or accounts office.
668,Fees,Please explain where can i find the latest fee structure for a university student?,The latest fee structure should be obtained from the official university website or accounts office.
669,Exams,Please explain when will the midterm exams start for a university student?,Midterm dates are published in the university's academic calendar or examination schedule.
670,Exams,Please explain when will the final exams start for a university student?,Final examination dates are announced by the examination office and academic calendar.
671,Exams,Please explain where can i find the exam timetable for a university student?,"Check the student portal, examination office notice board, or official university announcements."
672,Exams,Please explain what should i bring to the exam for a university student?,Bring the identification and stationery permitted by the university examination rules.
673,Exams,Please explain what happens if i miss an exam for a university student?,"If you miss an exam, follow the university's rules for absence and contact the examination office promptly."
674,Exams,Please explain can i apply for a re-examination for a university student?,Re-examination opportunities depend on the university's examination regulations.
675,Exams,Please explain how can i apply for a supplementary exam for a university student?,Apply through the examination office according to the current supplementary examination procedure.
676,Exams,Please explain what are the examination rules for a university student?,Examination rules are defined by the university examination regulations and should be followed by all students.
677,Exams,Please explain when are exam results announced for a university student?,Results are announced after evaluation and approval according to the university's examination process.
678,Exams,Please explain how can i check my exam result for a university student?,Check your result through the official student portal or examination office.
679,Attendance,Please explain what is the minimum attendance requirement for a university student?,The minimum attendance requirement is defined by the university's academic regulations. Check the current attendance policy.
680,Attendance,Please explain how can i check my attendance for a university student?,Attendance can usually be viewed through the student portal or confirmed by the relevant department.
681,Attendance,Please explain what happens if my attendance is below the required percentage for a university student?,Students below the required attendance level may face academic restrictions according to university policy.
682,Attendance,Please explain can i request attendance correction for a university student?,"If your attendance record is incorrect, contact the course instructor or department office with supporting evidence."
683,Attendance,Please explain does medical leave affect attendance for a university student?,Medical leave is handled according to university leave and attendance regulations.
684,Attendance,Please explain how do i apply for academic leave for a university student?,Submit a leave application through the procedure specified by your department or student affairs office.
685,Attendance,Please explain can attendance be marked late for a university student?,Late attendance rules depend on the instructor and university attendance policy.
686,Attendance,Please explain who maintains the attendance record for a university student?,Attendance is normally recorded by course instructors and maintained through the department or academic system.
687,Attendance,Please explain can i attend another section to make up attendance for a university student?,Section changes or make-up attendance are allowed only when approved under university policy.
688,Attendance,Please explain why is my attendance missing from the portal for a university student?,Contact the course instructor or department if attendance has not been updated or appears incorrect.
689,Scholarships,Please explain what scholarships are available for students for a university student?,"Available scholarships depend on the university, government programs, donors, and current scholarship announcements."
690,Scholarships,Please explain how can i apply for a scholarship for a university student?,Follow the application procedure published by the university scholarship or financial aid office.
691,Scholarships,Please explain what are the scholarship eligibility criteria for a university student?,"Eligibility varies by scholarship and may consider academic performance, financial need, or other criteria."
692,Scholarships,Please explain when is the scholarship application deadline for a university student?,Check the current scholarship announcement for the exact deadline.
693,Scholarships,Please explain is there a merit scholarship for a university student?,"Many universities offer merit-based scholarships, but availability and criteria should be confirmed from the current official notice."
694,Scholarships,Please explain is financial assistance available for a university student?,"Financial assistance may be available through scholarships, aid programs, or fee support. Contact the financial aid office."
695,Scholarships,Please explain can international students apply for scholarships for a university student?,Eligibility depends on the scholarship rules and university policy.
696,Scholarships,Please explain can i receive more than one scholarship for a university student?,Receiving multiple scholarships depends on the terms and conditions of the specific programs.
697,Scholarships,Please explain how will i know if my scholarship is approved for a university student?,"Scholarship decisions are normally communicated through the student portal, official notice, or scholarship office."
698,Scholarships,Please explain who should i contact about scholarships for a university student?,Contact the university scholarship or financial aid office for current information.
699,Library,Please explain what are the library opening hours for a university student?,Library hours vary by campus and semester. Check the latest library schedule.
700,Library,Please explain how can i get a library card for a university student?,Library membership is normally activated through the university library according to its registration procedure.
701,Library,Please explain how many books can i borrow for a university student?,The borrowing limit depends on student status and library policy.
702,Library,Please explain how long can i keep a borrowed book for a university student?,The loan period is determined by the library's current borrowing policy.
703,Library,Please explain can i renew a borrowed book for a university student?,Books may be renewable if they are eligible and not reserved by another user.
704,Library,Please explain what happens if i return a book late for a university student?,Late returns may result in fines or borrowing restrictions according to library policy.
705,Library,Please explain how can i search for a book for a university student?,Use the university library catalog or ask library staff for assistance.
706,Library,Please explain can i access online journals for a university student?,"If the university subscribes to online databases, students can access them through the library's authorized services."
707,Library,Please explain can i reserve a book for a university student?,Book reservation availability depends on the library system and current policy.
708,Library,Please explain where is the university library for a university student?,The library location depends on the campus. Check the official campus map or library information.
709,Hostel,Please explain how can i apply for university hostel accommodation for a university student?,Submit a hostel application through the university's approved hostel application procedure.
710,Hostel,Please explain who is eligible for hostel accommodation for a university student?,"Eligibility depends on university hostel rules, available rooms, and student status."
711,Hostel,Please explain what is the hostel fee for a university student?,Hostel fees vary by campus and room type. Check the latest official hostel fee schedule.
712,Hostel,Please explain when is hostel admission open for a university student?,Hostel application dates are announced by the university hostel administration.
713,Hostel,Please explain can i choose my roommate for a university student?,Roommate selection depends on hostel policy and room availability.
714,Hostel,Please explain what documents are required for hostel admission for a university student?,"Required documents may include student identification, admission proof, photographs, and other documents specified by hostel administration."
715,Hostel,Please explain what are the hostel rules for a university student?,"Hostel rules cover residence, discipline, visitors, safety, and other matters. Follow the current official hostel regulations."
716,Hostel,Please explain can i leave the hostel during the semester for a university student?,Hostel withdrawal or room cancellation follows the hostel administration's procedure.
717,Hostel,Please explain how can i report a hostel problem for a university student?,"Report hostel maintenance, safety, or administrative issues to the hostel warden or designated office."
718,Hostel,Please explain are meals provided in the hostel for a university student?,Meal facilities depend on the hostel and campus arrangements. Check the current hostel information.
719,Student Portal,Please explain how do i log in to the student portal for a university student?,Use the credentials provided by the university and access the official student portal.
720,Student Portal,Please explain i forgot my student portal password. what should i do for a university student?,Use the portal's password recovery option or contact the university IT/help desk.
721,Student Portal,Please explain how can i change my portal password for a university student?,Use the account settings or password-change option in the official student portal.
722,Student Portal,Please explain how can i update my profile information for a university student?,"Update profile information through the student portal if editing is enabled, or contact the relevant office."
723,Student Portal,Please explain why can't i log in to the student portal for a university student?,"Check your credentials and internet connection. If the problem continues, contact the university IT support office."
724,Student Portal,Please explain how can i register for courses online for a university student?,Use the course registration feature in the student portal during the announced registration period.
725,Student Portal,Please explain how can i download my fee voucher for a university student?,Log in to the student portal and use the fee or finance section if the university provides electronic vouchers.
726,Student Portal,Please explain how can i download my transcript for a university student?,"If electronic transcripts are supported, request or download them through the student portal; otherwise contact the registrar."
727,Student Portal,Please explain how can i see my registered courses for a university student?,Open the registered courses or enrollment section of the student portal.
728,Student Portal,Please explain how can i contact technical support for the portal for a university student?,Contact the university IT help desk through the official support channel.
729,Courses,Please explain how do i register for a course for a university student?,Course registration is completed through the university's registration system during the announced registration period.
730,Courses,Please explain can i drop a course for a university student?,Course withdrawal is allowed only within the period and conditions defined by academic regulations.
731,Courses,Please explain can i add a course after registration for a university student?,Adding a course after registration depends on the add/drop deadline and departmental approval.
732,Courses,Please explain what is a prerequisite course for a university student?,A prerequisite is a course that must normally be completed before taking another course.
733,Courses,Please explain how can i find my course prerequisites for a university student?,Check the official degree plan or course catalog for prerequisite requirements.
734,Courses,Please explain can i repeat a failed course for a university student?,Course repetition is generally governed by the university's academic regulations.
735,Courses,Please explain what is a credit hour for a university student?,A credit hour is a unit used to measure the academic workload of a course.
736,Courses,Please explain how many courses can i take in a semester for a university student?,"The allowed course load depends on the program, semester, and university academic regulations."
737,Courses,Please explain can i take an extra course for a university student?,An extra course may require academic approval and must comply with the maximum credit-hour policy.
738,Courses,Please explain what happens if i withdraw from a course for a university student?,The academic and transcript consequences depend on the university's withdrawal policy.
739,Faculty,Please explain how can i contact my course instructor for a university student?,"Use the official university email, learning platform, department office, or other approved communication channel."
740,Faculty,Please explain where can i find faculty office hours for a university student?,"Faculty office hours may be listed on the department website, student portal, or department notice board."
741,Faculty,Please explain who is the head of my department for a university student?,The current department head should be confirmed from the official university department page.
742,Faculty,Please explain how can i meet a faculty member for a university student?,Contact the faculty member through the official channel and request an appointment during available hours.
743,Faculty,Please explain how can i submit an assignment to my instructor for a university student?,Submit assignments using the platform or method specified by the instructor.
744,Faculty,Please explain what should i do if i have an academic issue with a course for a university student?,"First discuss the issue with the course instructor, then follow the department's academic complaint procedure if necessary."
745,Faculty,Please explain how can i request a recommendation letter for a university student?,Ask the faculty member according to the university's recommendation or reference procedure and provide the required information.
746,Faculty,Please explain can i change my academic advisor for a university student?,Advisor changes depend on departmental policy and approval.
747,Faculty,Please explain who is my academic advisor for a university student?,Your academic advisor can be identified through the student portal or department office.
748,Faculty,Please explain how can i contact the department office for a university student?,Use the department's official contact information listed on the university website.
749,Campus Facilities,Please explain does the university have a computer lab for a university student?,Check the campus facilities list to confirm available computer labs and their locations.
750,Campus Facilities,Please explain does the university provide wi-fi for a university student?,University Wi-Fi availability and access rules depend on the campus IT policy.
751,Campus Facilities,Please explain how can i access campus wi-fi for a university student?,Use the credentials and connection instructions provided by the university IT department.
752,Campus Facilities,Please explain is there a cafeteria on campus for a university student?,Cafeteria availability depends on the campus. Check the official campus facilities information.
753,Campus Facilities,Please explain does the campus have a sports facility for a university student?,Sports facilities vary by campus and can be confirmed through student affairs or campus facilities information.
754,Campus Facilities,Please explain is there a medical center on campus for a university student?,Check the campus facilities or student services information for available medical services.
755,Campus Facilities,Please explain where can i report a maintenance issue for a university student?,Report maintenance problems to the campus facilities or administration office through the approved channel.
756,Campus Facilities,Please explain is parking available for students for a university student?,Student parking availability and permits depend on campus parking rules.
757,Campus Facilities,Please explain how can i get a student id card for a university student?,Student ID cards are issued through the university's designated student services or administration office.
758,Campus Facilities,Please explain what should i do if i lose my student id card for a university student?,Report the lost card immediately to student services and follow the replacement procedure.
759,Graduation,Please explain what are the graduation requirements for a university student?,Graduation requirements are defined by the degree plan and university academic regulations.
760,Graduation,Please explain how can i apply for graduation for a university student?,Submit a graduation application through the registrar or student portal according to the announced procedure.
761,Graduation,Please explain when should i apply for graduation for a university student?,Apply during the graduation application period announced by the registrar.
762,Graduation,Please explain how can i check whether i have completed my degree requirements for a university student?,Review your degree audit or contact your academic advisor or registrar.
763,Graduation,Please explain what documents are needed for graduation clearance for a university student?,"Required documents vary by university and may include clearance forms, identification, and financial or departmental clearance."
764,Graduation,Please explain when is the graduation ceremony for a university student?,Ceremony dates are announced by the university through official notices.
765,Graduation,Please explain how can i get my degree certificate for a university student?,Degree certificates are issued by the registrar or examination authority after completion of all requirements.
766,Graduation,Please explain can i attend graduation if my clearance is incomplete for a university student?,Participation depends on the university's graduation and clearance rules.
767,Graduation,Please explain how can i request a duplicate degree for a university student?,Follow the registrar's procedure for replacement or duplicate degree documents.
768,Graduation,Please explain how can i obtain an official transcript after graduation for a university student?,Request an official transcript through the registrar or examination office.
769,Rules & Policies,Please explain where can i find university rules for a university student?,"University rules are normally published in official regulations, student handbooks, or the university website."
770,Rules & Policies,Please explain what is the student code of conduct for a university student?,The student code of conduct defines expected academic and disciplinary behavior.
771,Rules & Policies,Please explain what happens if a student violates university rules for a university student?,Disciplinary action depends on the nature of the violation and the university's disciplinary regulations.
772,Rules & Policies,Please explain how can i submit a complaint for a university student?,Use the official complaint or grievance procedure provided by student affairs or the relevant office.
773,Rules & Policies,Please explain how can i appeal an academic decision for a university student?,Follow the university's formal academic appeal procedure within the specified deadline.
774,Rules & Policies,Please explain what is the plagiarism policy for a university student?,Students must follow the university's academic integrity and plagiarism policy.
775,Rules & Policies,Please explain what is considered academic misconduct for a university student?,"Academic misconduct may include plagiarism, cheating, unauthorized collaboration, or other violations defined by university regulations."
776,Rules & Policies,Please explain can students use mobile phones during exams for a university student?,Mobile phone use during exams is governed by examination rules and may be prohibited.
777,Rules & Policies,Please explain what is the dress code for a university student?,"Dress requirements, if any, are defined by the university or campus policy."
778,Rules & Policies,Please explain where can i report a disciplinary concern for a university student?,"Report concerns through the university's designated student affairs, discipline, or complaint channel."
779,Timetable,Please explain where can i find my class timetable for a university student?,"Check the student portal, department notice board, or official timetable announcement."
780,Timetable,Please explain how can i know my classroom for a university student?,Classroom assignments are usually shown in the timetable or announced by the department.
781,Timetable,Please explain can the timetable change during the semester for a university student?,"Yes, timetable changes may occur due to academic or administrative requirements. Check official announcements."
782,Timetable,Please explain how can i report a timetable conflict for a university student?,Report schedule conflicts to the department or academic office promptly.
783,Timetable,Please explain how can i find the timetable for my department for a university student?,Check the department's official timetable announcement or student portal.
784,Timetable,Please explain what should i do if two classes are scheduled at the same time for a university student?,Contact the department or academic office to report the conflict and request guidance.
785,Timetable,Please explain where can i find room numbers for a university student?,Room numbers are normally listed with course schedules or posted by the department.
786,Timetable,Please explain how do i know if a class has been cancelled for a university student?,"Check official university announcements, the student portal, or communication from the instructor."
787,Timetable,Please explain can i change my class section for a university student?,Section changes depend on available seats and department approval.
788,Timetable,Please explain when is the class schedule published for a university student?,The schedule is normally published before the start of the academic term.
789,General Student Services,Please explain how can i contact student affairs for a university student?,Use the official student affairs office contact details published by the university.
790,General Student Services,Please explain where is the registrar office for a university student?,The registrar office location depends on the campus. Check the official campus directory.
791,General Student Services,Please explain what does the registrar office handle for a university student?,"The registrar commonly handles academic records, enrollment, transcripts, and graduation documentation."
792,General Student Services,Please explain how can i request an enrollment certificate for a university student?,Request an enrollment certificate through the registrar or student portal according to university procedure.
793,General Student Services,Please explain how can i request a bonafide student certificate for a university student?,Apply through the registrar or designated student services office.
794,General Student Services,Please explain how can i change my personal information for a university student?,Submit an update request through the student portal or registrar with required supporting documents.
795,General Student Services,Please explain how can i request a transcript for a university student?,Submit a transcript request through the registrar or examination office.
796,General Student Services,Please explain how can i get an official university letter for a university student?,Request the required letter from the relevant administrative office using the university's official procedure.
797,General Student Services,Please explain what is the university contact number for a university student?,Use the current official contact number published on the university website.
798,General Student Services,Please explain what is the university email address for a university student?,Use the current official email address published by the university.
799,Admission,Please explain what are the admission requirements for bs computer science (information for students) for current students for a university student?,Admission requirements for BS Computer Science depend on the university's current eligibility policy. Please check the official admissions requirements.
800,Admission,Please explain what are the admission requirements for bs artificial intelligence (information for students) for current students for a university student?,Admission requirements for BS Artificial Intelligence depend on the university's current eligibility policy. Please check the official admissions requirements.
801,Admission,Please explain what are the admission requirements for bs software engineering (information for students) for current students for a university student?,Admission requirements for BS Software Engineering depend on the university's current eligibility policy. Please check the official admissions requirements.
802,Admission,Please explain what are the admission requirements for bs information technology (information for students) for current students for a university student?,Admission requirements for BS Information Technology depend on the university's current eligibility policy. Please check the official admissions requirements.
803,Admission,Please explain what are the admission requirements for bba (information for students) for current students for a university student?,Admission requirements for BBA depend on the university's current eligibility policy. Please check the official admissions requirements.
804,Admission,Please explain what are the admission requirements for mba (information for students) for current students for a university student?,Admission requirements for MBA depend on the university's current eligibility policy. Please check the official admissions requirements.
805,Admission,Please explain how can i apply for admission to bs computer science (information for students) for the current academic year for a university student?,You can apply for BS Computer Science through the university's official admission portal or admissions office.
806,Admission,Please explain what is the procedure to apply for admission to bs artificial intelligence for the current academic year for a university student?,You can apply for BS Artificial Intelligence through the university's official admission portal or admissions office.
807,Admission,Please explain how can i apply for admission to bs software engineering (information for students) for the current academic year for a university student?,You can apply for BS Software Engineering through the university's official admission portal or admissions office.
808,Admission,Please explain how can i apply for admission to bs information technology (information for students) for the current academic year for a university student?,You can apply for BS Information Technology through the university's official admission portal or admissions office.
809,Admission,Please explain how can i apply for admission to bba (information for students) for the current academic year for a university student?,You can apply for BBA through the university's official admission portal or admissions office.
810,Admission,Please explain how can i apply for admission to mba (information for students) for the current academic year for a university student?,You can apply for MBA through the university's official admission portal or admissions office.
811,Admission,Please explain when does admission open for bs computer science (information for students) according to university policy for a university student?,Admission opening dates vary by university and intake. Please check the latest official admission schedule.
812,Admission,Please explain when does admission open for bs artificial intelligence (information for students) according to university policy for a university student?,Admission opening dates vary by university and intake. Please check the latest official admission schedule.
813,Admission,Please explain when does admission open for bs software engineering (information for students) according to university policy for a university student?,Admission opening dates vary by university and intake. Please check the latest official admission schedule.
814,Admission,Please explain when does admission open for bs information technology (information for students) according to university policy for a university student?,Admission opening dates vary by university and intake. Please check the latest official admission schedule.
815,Admission,Please explain when does admission open for bba (information for students) according to university policy for a university student?,Admission opening dates vary by university and intake. Please check the latest official admission schedule.
816,Admission,Please explain when does admission open for mba (information for students) according to university policy for a university student?,Admission opening dates vary by university and intake. Please check the latest official admission schedule.
817,Admission,Please explain what documents are required for admission (information for students) online for a university student?,"Common documents include academic certificates, identification documents, photographs, and the completed application form. The university may require additional documents."
818,Admission,Please explain can i apply online for admission (information for students) online for a university student?,Most universities provide an online admission application system. Please use the official university admission portal.
819,Admission,Please explain what is the admission deadline (information for students) online for a university student?,The admission deadline depends on the program and intake. Please check the current official admission notice.
820,Admission,Please explain can i apply after the deadline (information for students) online for a university student?,Late applications are accepted only if the university announces an extension. Check the latest official notice.
821,Admission,Please explain how can i check my admission application status (information for students) online for a university student?,Check your application status through the university admission portal or contact the admissions office.
822,Admission,Please explain what is the admission fee (information for students) online for a university student?,The admission or application fee varies by university and program. Check the current fee schedule.
823,Admission,Please explain can i change my selected program after applying (information for students) through the student portal for a university student?,Program changes may be possible before a specified deadline. Contact the admissions office for the current procedure.
824,Programs,Please explain what programs does the university offer (information for students) through the student portal for a university student?,The university offers programs listed in its current academic prospectus. Check the official programs and departments list.
825,Programs,Please explain does the university offer bs computer science (information for students) through the student portal for a university student?,Please check the university's current programs list to confirm whether BS Computer Science is offered.
826,Programs,Please explain does the university offer bs artificial intelligence (information for students) through the student portal for a university student?,Please check the university's current programs list to confirm whether BS Artificial Intelligence is offered.
827,Programs,Please explain does the university offer bs software engineering (information for students) through the student portal for a university student?,Please check the university's current programs list to confirm whether BS Software Engineering is offered.
828,Programs,Please explain does the university offer bs information technology (information for students) through the student portal for a university student?,Please check the university's current programs list to confirm whether BS Information Technology is offered.
829,Programs,Please explain does the university offer bba (information for students) at the university for a university student?,Please check the university's current programs list to confirm whether BBA is offered.
830,Programs,Please explain does the university offer mba (information for students) at the university for a university student?,Please check the university's current programs list to confirm whether MBA is offered.
831,Programs,Please explain how long is the bs computer science program (information for students) at the university for a university student?,The duration of BS Computer Science depends on the university's approved curriculum and academic regulations.
832,Programs,Please explain how long is the bs artificial intelligence program (information for students) at the university for a university student?,The duration of BS Artificial Intelligence depends on the university's approved curriculum and academic regulations.
833,Programs,Please explain how long is the bs software engineering program (information for students) at the university for a university student?,The duration of BS Software Engineering depends on the university's approved curriculum and academic regulations.
834,Programs,Please explain how long is the bs information technology program (information for students) at the university for a university student?,The duration of BS Information Technology depends on the university's approved curriculum and academic regulations.
835,Programs,Please explain how long is the bba program (information for students) for current students for a university student?,The duration of BBA depends on the university's approved curriculum and academic regulations.
836,Programs,Please explain how long is the mba program (information for students) for current students for a university student?,The duration of MBA depends on the university's approved curriculum and academic regulations.
837,Programs,Please explain what subjects are included in bs computer science (information for students) for current students for a university student?,The subjects for BS Computer Science are listed in the official curriculum or degree plan.
838,Programs,Please explain what subjects are included in bs artificial intelligence (information for students) for current students for a university student?,The subjects for BS Artificial Intelligence are listed in the official curriculum or degree plan.
839,Programs,Please explain what subjects are included in bs software engineering (information for students) for current students for a university student?,The subjects for BS Software Engineering are listed in the official curriculum or degree plan.
840,Programs,Please explain what subjects are included in bs information technology (information for students) for current students for a university student?,The subjects for BS Information Technology are listed in the official curriculum or degree plan.
841,Programs,Please explain what subjects are included in bba (information for students) for the current academic year for a university student?,The subjects for BBA are listed in the official curriculum or degree plan.
842,Programs,Please explain what subjects are included in mba (information for students) for the current academic year for a university student?,The subjects for MBA are listed in the official curriculum or degree plan.
843,Programs,Please explain is bs computer science available in morning classes (information for students) for the current academic year for a university student?,Class timing depends on the university and department schedule. Check the current timetable.
844,Programs,Please explain is bs artificial intelligence available in morning classes (information for students) for the current academic year for a university student?,Class timing depends on the university and department schedule. Check the current timetable.
845,Programs,Please explain is bs software engineering available in morning classes (information for students) for the current academic year for a university student?,Class timing depends on the university and department schedule. Check the current timetable.
846,Programs,Please explain is bs information technology available in morning classes (information for students) for the current academic year for a university student?,Class timing depends on the university and department schedule. Check the current timetable.
847,Programs,Please explain is bba available in morning classes (information for students) according to university policy for a university student?,Class timing depends on the university and department schedule. Check the current timetable.
848,Programs,Please explain is mba available in morning classes (information for students) according to university policy for a university student?,Class timing depends on the university and department schedule. Check the current timetable.
849,Programs,Please explain is bs computer science available in evening classes (information for students) according to university policy for a university student?,Evening availability depends on the department's current schedule.
850,Programs,Please explain is bs artificial intelligence available in evening classes (information for students) according to university policy for a university student?,Evening availability depends on the department's current schedule.
851,Programs,Please explain is bs software engineering available in evening classes (information for students) according to university policy for a university student?,Evening availability depends on the department's current schedule.
852,Programs,Please explain is bs information technology available in evening classes (information for students) according to university policy for a university student?,Evening availability depends on the department's current schedule.
853,Programs,Please explain is bba available in evening classes (information for students) online for a university student?,Evening availability depends on the department's current schedule.
854,Programs,Please explain is mba available in evening classes (information for students) online for a university student?,Evening availability depends on the department's current schedule.
855,Programs,Please explain what is the scope of bs computer science (information for students) online for a university student?,"The scope of BS Computer Science depends on industry demand, skills, specialization, and career opportunities."
856,Programs,Please explain could you explain the scope of bs artificial intelligence online for a university student?,"The scope of BS Artificial Intelligence depends on industry demand, skills, specialization, and career opportunities."
857,Programs,Please explain what is the scope of bs software engineering (information for students) online for a university student?,"The scope of BS Software Engineering depends on industry demand, skills, specialization, and career opportunities."
858,Programs,Please explain what is the scope of bs information technology (information for students) online for a university student?,"The scope of BS Information Technology depends on industry demand, skills, specialization, and career opportunities."
859,Programs,Please explain what is the scope of bba (information for students) through the student portal for a university student?,"The scope of BBA depends on industry demand, skills, specialization, and career opportunities."
860,Programs,Please explain what is the scope of mba (information for students) through the student portal for a university student?,"The scope of MBA depends on industry demand, skills, specialization, and career opportunities."
861,Programs,Please explain what careers are available after bs computer science (information for students) through the student portal for a university student?,Career options after BS Computer Science depend on the skills and specialization you develop. Consult the program's official career information.
862,Programs,Please explain what careers are available after bs artificial intelligence (information for students) through the student portal for a university student?,Career options after BS Artificial Intelligence depend on the skills and specialization you develop. Consult the program's official career information.
863,Programs,Please explain what careers are available after bs software engineering (information for students) through the student portal for a university student?,Career options after BS Software Engineering depend on the skills and specialization you develop. Consult the program's official career information.
864,Programs,Please explain what careers are available after bs information technology (information for students) through the student portal for a university student?,Career options after BS Information Technology depend on the skills and specialization you develop. Consult the program's official career information.
865,Programs,Please explain what careers are available after bba (information for students) at the university for a university student?,Career options after BBA depend on the skills and specialization you develop. Consult the program's official career information.
866,Programs,Please explain what careers are available after mba (information for students) at the university for a university student?,Career options after MBA depend on the skills and specialization you develop. Consult the program's official career information.
867,Programs,Please explain can i change my major (information for students) at the university for a university student?,Major or program changes are subject to university academic regulations and available seats.
868,Programs,Please explain where can i find the degree curriculum (information for students) at the university for a university student?,"The official curriculum can usually be found on the university website, department office, or academic portal."
869,Fees,Please explain what is the tuition fee for bs computer science (information for students) at the university for a university student?,The tuition fee for BS Computer Science should be confirmed from the university's latest official fee structure.
870,Fees,Please explain what is the tuition fee for bs artificial intelligence (information for students) at the university for a university student?,The tuition fee for BS Artificial Intelligence should be confirmed from the university's latest official fee structure.
871,Fees,Please explain what is the tuition fee for bs software engineering (information for students) for current students for a university student?,The tuition fee for BS Software Engineering should be confirmed from the university's latest official fee structure.
872,Fees,Please explain what is the tuition fee for bs information technology (information for students) for current students for a university student?,The tuition fee for BS Information Technology should be confirmed from the university's latest official fee structure.
873,Fees,Please explain what is the tuition fee for bba (information for students) for current students for a university student?,The tuition fee for BBA should be confirmed from the university's latest official fee structure.
874,Fees,Please explain could you explain the tuition fee for mba for current students for a university student?,The tuition fee for MBA should be confirmed from the university's latest official fee structure.
875,Fees,Please explain how can i pay my semester fee (information for students) for current students for a university student?,"Use the payment methods authorized by the university, such as its online portal or designated bank/payment channels."
876,Fees,Please explain what is the last date for fee payment (information for students) for current students for a university student?,The fee payment deadline is published in the academic calendar or fee notice.
877,Fees,Please explain is there a late fee for delayed payment (information for students) for the current academic year for a university student?,A late payment charge may apply according to university regulations. Check the latest fee policy.
878,Fees,Please explain can i pay my fee in installments (information for students) for the current academic year for a university student?,Installment options depend on university policy and may require approval from the relevant office.
879,Fees,Please explain how can i get my fee voucher (information for students) for the current academic year for a university student?,Fee vouchers are normally available through the student portal or accounts office.
880,Fees,Please explain what should i do if my fee payment is not updated (information for students) for the current academic year for a university student?,Keep your payment receipt and contact the accounts or finance office to have the transaction verified.
881,Fees,Please explain can i get a fee refund (information for students) for the current academic year for a university student?,Refund eligibility depends on the university's refund policy and the reason for withdrawal or cancellation.
882,Fees,Please explain how can i request a fee refund (information for students) for the current academic year for a university student?,Submit a refund request through the procedure specified by the university finance or accounts office.
883,Fees,Please explain where can i find the latest fee structure (information for students) according to university policy for a university student?,The latest fee structure should be obtained from the official university website or accounts office.
884,Exams,Please explain when will the midterm exams start (information for students) according to university policy for a university student?,Midterm dates are published in the university's academic calendar or examination schedule.
885,Exams,Please explain when will the final exams start (information for students) according to university policy for a university student?,Final examination dates are announced by the examination office and academic calendar.
886,Exams,Please explain where can i find the exam timetable (information for students) according to university policy for a university student?,"Check the student portal, examination office notice board, or official university announcements."
887,Exams,Please explain what should i bring to the exam (information for students) according to university policy for a university student?,Bring the identification and stationery permitted by the university examination rules.
888,Exams,Please explain what happens if i miss an exam (information for students) according to university policy for a university student?,"If you miss an exam, follow the university's rules for absence and contact the examination office promptly."
889,Exams,Please explain can i apply for a re-examination (information for students) online for a university student?,Re-examination opportunities depend on the university's examination regulations.
890,Exams,Please explain what is the procedure to apply for a supplementary exam online for a university student?,Apply through the examination office according to the current supplementary examination procedure.
891,Exams,Please explain what are the examination rules (information for students) online for a university student?,Examination rules are defined by the university examination regulations and should be followed by all students.
892,Exams,Please explain when are exam results announced (information for students) online for a university student?,Results are announced after evaluation and approval according to the university's examination process.
893,Exams,Please explain how can i check my exam result (information for students) online for a university student?,Check your result through the official student portal or examination office.
894,Attendance,Please explain what is the minimum attendance requirement (information for students) online for a university student?,The minimum attendance requirement is defined by the university's academic regulations. Check the current attendance policy.
895,Attendance,Please explain how can i check my attendance (information for students) through the student portal for a university student?,Attendance can usually be viewed through the student portal or confirmed by the relevant department.
896,Attendance,Please explain what happens if my attendance is below the required percentage (information for students) through the student portal for a university student?,Students below the required attendance level may face academic restrictions according to university policy.
897,Attendance,Please explain can i request attendance correction (information for students) through the student portal for a university student?,"If your attendance record is incorrect, contact the course instructor or department office with supporting evidence."
898,Attendance,Please explain does medical leave affect attendance (information for students) through the student portal for a university student?,Medical leave is handled according to university leave and attendance regulations.
899,Attendance,Please explain how do i apply for academic leave (information for students) through the student portal for a university student?,Submit a leave application through the procedure specified by your department or student affairs office.
900,Attendance,Please explain can attendance be marked late (information for students) through the student portal for a university student?,Late attendance rules depend on the instructor and university attendance policy.
901,Attendance,Please explain who maintains the attendance record (information for students) at the university for a university student?,Attendance is normally recorded by course instructors and maintained through the department or academic system.
902,Attendance,Please explain can i attend another section to make up attendance (information for students) at the university for a university student?,Section changes or make-up attendance are allowed only when approved under university policy.
903,Attendance,Please explain why is my attendance missing from the portal (information for students) at the university for a university student?,Contact the course instructor or department if attendance has not been updated or appears incorrect.
904,Scholarships,Please explain what scholarships are available for students (information for students) at the university for a university student?,"Available scholarships depend on the university, government programs, donors, and current scholarship announcements."
905,Scholarships,Please explain how can i apply for a scholarship (information for students) at the university for a university student?,Follow the application procedure published by the university scholarship or financial aid office.
906,Scholarships,Please explain what are the scholarship eligibility criteria (information for students) at the university for a university student?,"Eligibility varies by scholarship and may consider academic performance, financial need, or other criteria."
907,Scholarships,Please explain when is the scholarship application deadline (information for students) for current students for a university student?,Check the current scholarship announcement for the exact deadline.
908,Scholarships,Please explain is there a merit scholarship (information for students) for current students for a university student?,"Many universities offer merit-based scholarships, but availability and criteria should be confirmed from the current official notice."
909,Scholarships,Please explain is financial assistance available (information for students) for current students for a university student?,"Financial assistance may be available through scholarships, aid programs, or fee support. Contact the financial aid office."
910,Scholarships,Please explain can international students apply for scholarships (information for students) for current students for a university student?,Eligibility depends on the scholarship rules and university policy.
911,Scholarships,Please explain can i receive more than one scholarship (information for students) for current students for a university student?,Receiving multiple scholarships depends on the terms and conditions of the specific programs.
912,Scholarships,Please explain how will i know if my scholarship is approved (information for students) for current students for a university student?,"Scholarship decisions are normally communicated through the student portal, official notice, or scholarship office."
913,Scholarships,Please explain who should i contact about scholarships (information for students) for the current academic year for a university student?,Contact the university scholarship or financial aid office for current information.
914,Library,Please explain what are the library opening hours (information for students) for the current academic year for a university student?,Library hours vary by campus and semester. Check the latest library schedule.
915,Library,Please explain how can i get a library card (information for students) for the current academic year for a university student?,Library membership is normally activated through the university library according to its registration procedure.
916,Library,Please explain how many books can i borrow (information for students) for the current academic year for a university student?,The borrowing limit depends on student status and library policy.
917,Library,Please explain how long can i keep a borrowed book (information for students) for the current academic year for a university student?,The loan period is determined by the library's current borrowing policy.
918,Library,Please explain am i allowed to renew a borrowed book for the current academic year for a university student?,Books may be renewable if they are eligible and not reserved by another user.
919,Library,Please explain what happens if i return a book late (information for students) according to university policy for a university student?,Late returns may result in fines or borrowing restrictions according to library policy.
920,Library,Please explain what is the procedure to search for a book according to university policy for a university student?,Use the university library catalog or ask library staff for assistance.
921,Library,Please explain can i access online journals (information for students) according to university policy for a university student?,"If the university subscribes to online databases, students can access them through the library's authorized services."
922,Library,Please explain can i reserve a book (information for students) according to university policy for a university student?,Book reservation availability depends on the library system and current policy.
923,Library,Please explain where is the university library (information for students) according to university policy for a university student?,The library location depends on the campus. Check the official campus map or library information.
924,Hostel,Please explain how can i apply for university hostel accommodation (information for students) according to university policy for a university student?,Submit a hostel application through the university's approved hostel application procedure.
925,Hostel,Please explain who is eligible for hostel accommodation (information for students) online for a university student?,"Eligibility depends on university hostel rules, available rooms, and student status."
926,Hostel,Please explain what is the hostel fee (information for students) online for a university student?,Hostel fees vary by campus and room type. Check the latest official hostel fee schedule.
927,Hostel,Please explain when is hostel admission open (information for students) online for a university student?,Hostel application dates are announced by the university hostel administration.
928,Hostel,Please explain can i choose my roommate (information for students) online for a university student?,Roommate selection depends on hostel policy and room availability.
929,Hostel,Please explain what documents are required for hostel admission (information for students) online for a university student?,"Required documents may include student identification, admission proof, photographs, and other documents specified by hostel administration."
930,Hostel,Please explain what are the hostel rules (information for students) online for a university student?,"Hostel rules cover residence, discipline, visitors, safety, and other matters. Follow the current official hostel regulations."
931,Hostel,Please explain can i leave the hostel during the semester (information for students) through the student portal for a university student?,Hostel withdrawal or room cancellation follows the hostel administration's procedure.
932,Hostel,Please explain what is the procedure to report a hostel problem through the student portal for a university student?,"Report hostel maintenance, safety, or administrative issues to the hostel warden or designated office."
933,Hostel,Please explain are meals provided in the hostel (information for students) through the student portal for a university student?,Meal facilities depend on the hostel and campus arrangements. Check the current hostel information.
934,Student Portal,Please explain how do i log in to the student portal (information for students) through the student portal for a university student?,Use the credentials provided by the university and access the official student portal.
935,Student Portal,Please explain i forgot my student portal password. what should i do (information for students) through the student portal for a university student?,Use the portal's password recovery option or contact the university IT/help desk.
936,Student Portal,Please explain how can i change my portal password (information for students) through the student portal for a university student?,Use the account settings or password-change option in the official student portal.
937,Student Portal,Please explain how can i update my profile information (information for students) at the university for a university student?,"Update profile information through the student portal if editing is enabled, or contact the relevant office."
938,Student Portal,Please explain why can't i log in to the student portal (information for students) at the university for a university student?,"Check your credentials and internet connection. If the problem continues, contact the university IT support office."
939,Student Portal,Please explain how can i register for courses online (information for students) at the university for a university student?,Use the course registration feature in the student portal during the announced registration period.
940,Student Portal,Please explain how can i download my fee voucher (information for students) at the university for a university student?,Log in to the student portal and use the fee or finance section if the university provides electronic vouchers.
941,Student Portal,Please explain how can i download my transcript (information for students) at the university for a university student?,"If electronic transcripts are supported, request or download them through the student portal; otherwise contact the registrar."
942,Student Portal,Please explain how can i see my registered courses (information for students) at the university for a university student?,Open the registered courses or enrollment section of the student portal.
943,Student Portal,Please explain how can i contact technical support for the portal (information for students) for current students for a university student?,Contact the university IT help desk through the official support channel.
944,Courses,Please explain how do i register for a course (information for students) for current students for a university student?,Course registration is completed through the university's registration system during the announced registration period.
945,Courses,Please explain can i drop a course (information for students) for current students for a university student?,Course withdrawal is allowed only within the period and conditions defined by academic regulations.
946,Courses,Please explain can i add a course after registration (information for students) for current students for a university student?,Adding a course after registration depends on the add/drop deadline and departmental approval.
947,Courses,Please explain what is a prerequisite course (information for students) for current students for a university student?,A prerequisite is a course that must normally be completed before taking another course.
948,Courses,Please explain how can i find my course prerequisites (information for students) for current students for a university student?,Check the official degree plan or course catalog for prerequisite requirements.
949,Courses,Please explain can i repeat a failed course (information for students) for the current academic year for a university student?,Course repetition is generally governed by the university's academic regulations.
950,Courses,Please explain what is a credit hour (information for students) for the current academic year for a university student?,A credit hour is a unit used to measure the academic workload of a course.
951,Courses,Please explain how many courses can i take in a semester (information for students) for the current academic year for a university student?,"The allowed course load depends on the program, semester, and university academic regulations."
952,Courses,Please explain can i take an extra course (information for students) for the current academic year for a university student?,An extra course may require academic approval and must comply with the maximum credit-hour policy.
953,Courses,Please explain what happens if i withdraw from a course (information for students) for the current academic year for a university student?,The academic and transcript consequences depend on the university's withdrawal policy.
954,Faculty,Please explain how can i contact my course instructor (information for students) for the current academic year for a university student?,"Use the official university email, learning platform, department office, or other approved communication channel."
955,Faculty,Please explain where can i find faculty office hours (information for students) according to university policy for a university student?,"Faculty office hours may be listed on the department website, student portal, or department notice board."
956,Faculty,Please explain who is the head of my department (information for students) according to university policy for a university student?,The current department head should be confirmed from the official university department page.
957,Faculty,Please explain how can i meet a faculty member (information for students) according to university policy for a university student?,Contact the faculty member through the official channel and request an appointment during available hours.
958,Faculty,Please explain how can i submit an assignment to my instructor (information for students) according to university policy for a university student?,Submit assignments using the platform or method specified by the instructor.
959,Faculty,Please explain what should i do if i have an academic issue with a course (information for students) according to university policy for a university student?,"First discuss the issue with the course instructor, then follow the department's academic complaint procedure if necessary."
960,Faculty,Please explain how can i request a recommendation letter (information for students) according to university policy for a university student?,Ask the faculty member according to the university's recommendation or reference procedure and provide the required information.
961,Faculty,Please explain can i change my academic advisor (information for students) online for a university student?,Advisor changes depend on departmental policy and approval.
962,Faculty,Please explain who is my academic advisor (information for students) online for a university student?,Your academic advisor can be identified through the student portal or department office.
963,Faculty,Please explain how can i contact the department office (information for students) online for a university student?,Use the department's official contact information listed on the university website.
964,Campus Facilities,Please explain does the university have a computer lab (information for students) online for a university student?,Check the campus facilities list to confirm available computer labs and their locations.
965,Campus Facilities,Please explain does the university provide wi-fi (information for students) online for a university student?,University Wi-Fi availability and access rules depend on the campus IT policy.
966,Campus Facilities,Please explain how can i access campus wi-fi (information for students) online for a university student?,Use the credentials and connection instructions provided by the university IT department.
967,Campus Facilities,Please explain is there a cafeteria on campus (information for students) through the student portal for a university student?,Cafeteria availability depends on the campus. Check the official campus facilities information.
968,Campus Facilities,Please explain does the campus have a sports facility (information for students) through the student portal for a university student?,Sports facilities vary by campus and can be confirmed through student affairs or campus facilities information.
969,Campus Facilities,Please explain is there a medical center on campus (information for students) through the student portal for a university student?,Check the campus facilities or student services information for available medical services.
970,Campus Facilities,Please explain where can i report a maintenance issue (information for students) through the student portal for a university student?,Report maintenance problems to the campus facilities or administration office through the approved channel.
971,Campus Facilities,Please explain is parking available for students (information for students) through the student portal for a university student?,Student parking availability and permits depend on campus parking rules.
972,Campus Facilities,Please explain how can i get a student id card (information for students) through the student portal for a university student?,Student ID cards are issued through the university's designated student services or administration office.
973,Campus Facilities,Please explain what should i do if i lose my student id card (information for students) at the university for a university student?,Report the lost card immediately to student services and follow the replacement procedure.
974,Graduation,Please explain what are the graduation requirements (information for students) at the university for a university student?,Graduation requirements are defined by the degree plan and university academic regulations.
975,Graduation,Please explain how can i apply for graduation (information for students) at the university for a university student?,Submit a graduation application through the registrar or student portal according to the announced procedure.
976,Graduation,Please explain when should i apply for graduation (information for students) at the university for a university student?,Apply during the graduation application period announced by the registrar.
977,Graduation,Please explain how can i check whether i have completed my degree requirements (information for students) at the university for a university student?,Review your degree audit or contact your academic advisor or registrar.
978,Graduation,Please explain what documents are needed for graduation clearance (information for students) at the university for a university student?,"Required documents vary by university and may include clearance forms, identification, and financial or departmental clearance."
979,Graduation,Please explain when is the graduation ceremony (information for students) for current students for a university student?,Ceremony dates are announced by the university through official notices.
980,Graduation,Please explain what is the procedure to get my degree certificate for current students for a university student?,Degree certificates are issued by the registrar or examination authority after completion of all requirements.
981,Graduation,Please explain can i attend graduation if my clearance is incomplete (information for students) for current students for a university student?,Participation depends on the university's graduation and clearance rules.
982,Graduation,Please explain how can i request a duplicate degree (information for students) for current students for a university student?,Follow the registrar's procedure for replacement or duplicate degree documents.
983,Graduation,Please explain how can i obtain an official transcript after graduation (information for students) for current students for a university student?,Request an official transcript through the registrar or examination office.
984,Rules & Policies,Please explain where can i find university rules (information for students) for current students for a university student?,"University rules are normally published in official regulations, student handbooks, or the university website."
985,Rules & Policies,Please explain what is the student code of conduct (information for students) for the current academic year for a university student?,The student code of conduct defines expected academic and disciplinary behavior.
986,Rules & Policies,Please explain what happens if a student violates university rules (information for students) for the current academic year for a university student?,Disciplinary action depends on the nature of the violation and the university's disciplinary regulations.
987,Rules & Policies,Please explain how can i submit a complaint (information for students) for the current academic year for a university student?,Use the official complaint or grievance procedure provided by student affairs or the relevant office.
988,Rules & Policies,Please explain how can i appeal an academic decision (information for students) for the current academic year for a university student?,Follow the university's formal academic appeal procedure within the specified deadline.
989,Rules & Policies,Please explain what is the plagiarism policy (information for students) for the current academic year for a university student?,Students must follow the university's academic integrity and plagiarism policy.
990,Rules & Policies,Please explain what is considered academic misconduct (information for students) for the current academic year for a university student?,"Academic misconduct may include plagiarism, cheating, unauthorized collaboration, or other violations defined by university regulations."
991,Rules & Policies,Please explain can students use mobile phones during exams (information for students) according to university policy for a university student?,Mobile phone use during exams is governed by examination rules and may be prohibited.
992,Rules & Policies,Please explain what is the dress code (information for students) according to university policy for a university student?,"Dress requirements, if any, are defined by the university or campus policy."
993,Rules & Policies,Please explain where can i report a disciplinary concern (information for students) according to university policy for a university student?,"Report concerns through the university's designated student affairs, discipline, or complaint channel."
994,Timetable,Please explain where can i find my class timetable (information for students) according to university policy for a university student?,"Check the student portal, department notice board, or official timetable announcement."
995,Timetable,Please explain how can i know my classroom (information for students) according to university policy for a university student?,Classroom assignments are usually shown in the timetable or announced by the department.
996,Timetable,Please explain can the timetable change during the semester (information for students) according to university policy for a university student?,"Yes, timetable changes may occur due to academic or administrative requirements. Check official announcements."
997,Timetable,Please explain how can i report a timetable conflict (information for students) online for a university student?,Report schedule conflicts to the department or academic office promptly.
998,Timetable,Please explain what is the procedure to find the timetable for my department online for a university student?,Check the department's official timetable announcement or student portal.
999,Timetable,Please explain what should i do if two classes are scheduled at the same time (information for students) online for a university student?,Contact the department or academic office to report the conflict and request guidance.
1000,Timetable,Please explain where can i find room numbers (information for students) online for a university student?,Room numbers are normally listed with course schedules or posted by the department.


### `chatbot.py` — the NLP engine: preprocessing, TF-IDF, cosine similarity, CRUD

In [ ]:
%%writefile chatbot.py
"""
chatbot.py
-----------
Core logic for the Smart University FAQ Assistant (1000-question dataset).

Pipeline:
    User Question
        -> Text Cleaning (lowercase, remove punctuation)
        -> Tokenization (NLTK)
        -> Stopword Removal + Lemmatization + light synonym normalization
        -> TF-IDF Vectorization (fit on all FAQ questions, unigrams + bigrams)
        -> Cosine Similarity (query vector vs. every FAQ vector)
        -> Best match above threshold -> return its answer + confidence
        -> Below threshold -> fallback response

No UI code lives here, so this class is reusable by Gradio, a CLI, or an API.
It also exposes add / update / delete methods so an admin panel can manage
the dataset at runtime, with the search index rebuilding itself automatically.
"""

import re
import string
from pathlib import Path

import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def _ensure_nltk_data():
    resources = {
        "tokenizers/punkt": "punkt",
        "tokenizers/punkt_tab": "punkt_tab",
        "corpora/stopwords": "stopwords",
        "corpora/wordnet": "wordnet",
        "corpora/omw-1.4": "omw-1.4",
    }
    for path, pkg in resources.items():
        try:
            nltk.data.find(path)
        except LookupError:
            nltk.download(pkg, quiet=True)


_ensure_nltk_data()

# Canonical category order (matches the dataset), used for stable dropdowns.
CATEGORIES = [
    "Admission", "Programs", "Fees", "Exams", "Attendance", "Scholarships",
    "Library", "Hostel", "Student Portal", "Courses", "Faculty",
    "Campus Facilities", "Graduation", "Rules & Policies", "Timetable",
    "General Student Services",
]

FALLBACK_MESSAGE = "Sorry, I couldn't understand your question. Please try asking differently."
HUMAN_SUPPORT_MESSAGE = "If you still need help, please contact our support team."


class FAQChatbot:
    """A TF-IDF + cosine-similarity based FAQ matching chatbot."""

    # Light domain-synonym normalization so paraphrases line up with the
    # dataset's wording, e.g. "How much do I need to PAY" -> "fee".
    SYNONYMS = {
        "pay": "fee", "paying": "fee", "payment": "fee", "cost": "fee",
        "charge": "fee", "charges": "fee", "price": "fee", "amount": "fee",
        "tuition": "fee",
        "begin": "start", "starts": "start", "starting": "start", "open": "start",
        "commence": "start", "commences": "start",
        "call": "contact", "reach": "contact", "email": "contact", "number": "contact",
        "located": "location", "address": "location", "situated": "location",
        "dorm": "hostel", "dormitory": "hostel", "residence": "hostel",
        "course": "program", "courses": "program",
        "major": "program", "programme": "program", "programs": "program",
        "test": "exam", "examination": "exam", "quiz": "exam", "exams": "exam",
        "sign": "login", "signin": "login", "log": "login",
        "grade": "gpa", "grades": "gpa", "marks": "gpa", "result": "gpa", "results": "gpa",
        "money": "fee", "instalment": "installment", "installments": "installment",
        "admitted": "admission", "admit": "admission", "enrolled": "admission",
        "enroll": "admission", "enrollment": "admission", "join": "admission",
        "teacher": "faculty", "teachers": "faculty", "professor": "faculty",
        "professors": "faculty", "instructor": "faculty", "instructors": "faculty",
        "lecturer": "faculty", "lecturers": "faculty",
    }

    # Abbreviations that NLTK's lemmatizer mangles (e.g. "bs" -> "b" as if it
    # were a plural noun). Keep these tokens exactly as typed.
    PROTECTED_TOKENS = {"bs", "ms", "mba", "bba", "phd", "gpa", "cgpa", "id"}

    def __init__(self, faqs, similarity_threshold: float = 0.30):
        self.faqs = faqs  # list of dicts: id, category, question, answer
        self.similarity_threshold = similarity_threshold

        self.lemmatizer = WordNetLemmatizer()
        self.stop_words = set(stopwords.words("english"))

        self._build_index()

    # ------------------------------------------------------------------ #
    # Index management
    # ------------------------------------------------------------------ #
    def _build_index(self):
        """(Re)build the TF-IDF index for the current dataset."""
        self.questions = [item["question"] for item in self.faqs]
        self.cleaned_questions = [self._preprocess(q) for q in self.questions]

        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2))
        if self.cleaned_questions and any(self.cleaned_questions):
            self.tfidf_matrix = self.vectorizer.fit_transform(self.cleaned_questions)
        else:
            self.tfidf_matrix = None

    # ------------------------------------------------------------------ #
    # NLP preprocessing
    # ------------------------------------------------------------------ #
    def _preprocess(self, text: str) -> str:
        if not text:
            return ""
        text = text.lower().strip()
        text = re.sub(rf"[{re.escape(string.punctuation)}]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()

        tokens = word_tokenize(text)
        tokens = [
            tok if tok in self.PROTECTED_TOKENS else self.lemmatizer.lemmatize(tok)
            for tok in tokens
            if tok not in self.stop_words and tok.isalpha()
        ]
        tokens = [self.SYNONYMS.get(tok, tok) for tok in tokens]
        return " ".join(tokens)

    # ------------------------------------------------------------------ #
    # Matching
    # ------------------------------------------------------------------ #
    def get_best_match(self, user_question: str, category: str = None):
        """Returns dict: answer, matched_question, category, score, is_confident."""
        cleaned = self._preprocess(user_question)

        if not cleaned or self.tfidf_matrix is None:
            return self._fallback()

        query_vec = self.vectorizer.transform([cleaned])
        similarities = cosine_similarity(query_vec, self.tfidf_matrix).flatten()

        if category and category != "All":
            for i, item in enumerate(self.faqs):
                if item["category"] != category:
                    similarities[i] = -1.0

        best_idx = int(similarities.argmax())
        best_score = float(similarities[best_idx])

        if best_score >= self.similarity_threshold:
            match = self.faqs[best_idx]
            return {
                "answer": match["answer"],
                "matched_question": match["question"],
                "category": match.get("category"),
                "score": best_score,
                "is_confident": True,
            }

        return self._fallback(best_idx, best_score)

    def _fallback(self, closest_idx=None, closest_score=0.0):
        return {
            "answer": FALLBACK_MESSAGE,
            "matched_question": self.faqs[closest_idx]["question"] if closest_idx is not None else None,
            "category": None,
            "score": closest_score,
            "is_confident": False,
        }

    def top_matches(self, user_question: str, k: int = 5):
        cleaned = self._preprocess(user_question)
        if not cleaned or self.tfidf_matrix is None:
            return []

        query_vec = self.vectorizer.transform([cleaned])
        similarities = cosine_similarity(query_vec, self.tfidf_matrix).flatten()
        top_idx = similarities.argsort()[::-1][:k]

        return [
            {
                "question": self.faqs[i]["question"],
                "answer": self.faqs[i]["answer"],
                "category": self.faqs[i]["category"],
                "score": float(similarities[i]),
            }
            for i in top_idx
        ]

    # ------------------------------------------------------------------ #
    # Admin CRUD
    # ------------------------------------------------------------------ #
    def _next_id(self):
        return (max((item["id"] for item in self.faqs), default=0)) + 1

    def add_faq(self, category, question, answer):
        new_item = {"id": self._next_id(), "category": category, "question": question, "answer": answer}
        self.faqs.append(new_item)
        self._build_index()
        return new_item

    def update_faq(self, faq_id, **fields):
        for item in self.faqs:
            if item["id"] == faq_id:
                for key in ("category", "question", "answer"):
                    if key in fields:
                        item[key] = fields[key]
                self._build_index()
                return item
        return None

    def delete_faq(self, faq_id):
        before = len(self.faqs)
        self.faqs = [item for item in self.faqs if item["id"] != faq_id]
        self._build_index()
        return len(self.faqs) < before

    def save_to_csv(self, path):
        pd.DataFrame(self.faqs)[["id", "category", "question", "answer"]].to_csv(path, index=False)


def load_faqs(path):
    """Load the FAQ dataset from a CSV file with columns: id, category, question, answer."""
    df = pd.read_csv(Path(path), encoding="utf-8-sig")
    df = df.dropna(subset=["question", "answer"])
    return df[["id", "category", "question", "answer"]].to_dict(orient="records")


# ---------------------------------------------------------------------- #
# Quick manual test: `python chatbot.py`
# ---------------------------------------------------------------------- #
if __name__ == "__main__":
    faqs = load_faqs("faq.csv")
    print(f"Loaded {len(faqs)} FAQs across {len(set(f['category'] for f in faqs))} categories")

    bot = FAQChatbot(faqs)

    test_questions = [
        "What do I need to get admitted into BS Computer Science?",
        "How much does the Computer Science degree cost?",
        "when do midterm exams begin",
        "I can't remember my student portal password",
        "how many books can I take from the library",
        "is hostel available for girls",
        "what happens if I break university rules",
        "who is teaching in the computer science department",
    ]

    for q in test_questions:
        result = bot.get_best_match(q)
        print(f"\nUser: {q}")
        print(f"Matched [{result['category']}]: {result['matched_question']}")
        print(f"Score: {result['score']:.3f}  |  Confident: {result['is_confident']}")
        print(f"Bot: {result['answer']}")


### `voice_utils.py` — speech-to-text and text-to-speech helpers

In [ ]:
%%writefile voice_utils.py
"""
voice_utils.py
---------------
Optional voice features for the Smart FAQ Assistant:
    - transcribe_audio(): speech -> text (needs internet; uses Google's free
      Web Speech API via the `speech_recognition` library)
    - synthesize_speech(): text -> speech (needs internet; uses gTTS, which
      calls Google Translate's TTS endpoint)

Both are wrapped in try/except and return (result, error_message) so the
calling UI can degrade gracefully (e.g. on a machine with no internet access,
or no microphone) instead of crashing.
"""

import io

LANG_CODE_STT = {"en": "en-US", "ur": "ur-PK", "hi": "hi-IN"}
LANG_CODE_TTS = {"en": "en", "ur": "ur", "hi": "hi"}


def transcribe_audio(audio_path: str, language: str = "en"):
    """
    Convert a recorded audio file (e.g. path from gr.Audio(type='filepath'))
    to text. Returns (text, error). Exactly one of the two will be None.
    """
    try:
        import speech_recognition as sr
    except ImportError:
        return None, "The 'SpeechRecognition' package is not installed."

    if not audio_path:
        return None, "No audio received. Please record a question first."

    recognizer = sr.Recognizer()
    try:
        with sr.AudioFile(audio_path) as source:
            audio = recognizer.record(source)
    except Exception as e:
        return None, f"Could not read the audio: {e}"

    lang_code = LANG_CODE_STT.get(language, "en-US")
    try:
        text = recognizer.recognize_google(audio, language=lang_code)
        return text, None
    except sr.UnknownValueError:
        return None, "Sorry, I couldn't make out what you said. Please try again."
    except sr.RequestError as e:
        return None, f"Speech recognition service unavailable (needs internet): {e}"
    except Exception as e:
        return None, f"Speech recognition failed: {e}"


def synthesize_speech(text: str, language: str = "en"):
    """
    Convert text to speech audio bytes (MP3) using gTTS.
    Returns (audio_bytes, error). Exactly one of the two will be None.
    """
    try:
        from gtts import gTTS
    except ImportError:
        return None, "The 'gTTS' package is not installed."

    lang_code = LANG_CODE_TTS.get(language, "en")
    try:
        tts = gTTS(text=text, lang=lang_code)
        buf = io.BytesIO()
        tts.write_to_fp(buf)
        buf.seek(0)
        return buf.read(), None
    except Exception as e:
        return None, f"Text-to-speech unavailable (needs internet): {e}"


### `app.py` — the Gradio UI: Chat tab + Admin Panel tab

In [ ]:
%%writefile app.py
"""
app.py
-------
Smart University AI FAQ Chatbot — Gradio edition, trained on the
1,000-question dataset (faq.csv).

Features:
  1. Chat interface (gr.Chatbot, separate user/bot bubbles)
  2. FAQ matching via TF-IDF + cosine similarity (chatbot.py)
  3. NLP preprocessing (lowercase, cleanup, tokenize, TF-IDF)
  4. Confidence score shown per answer
  5. Unknown-question fallback + human-support message
  6. FAQ categories (16 categories from the dataset)
  7. Suggested/quick questions (buttons, filterable by category)
  8. Chat history with Clear option
  9. Admin panel: add / edit / delete / view FAQs, persist to disk
  10. Voice input (mic -> speech-to-text)
  11. Text-to-speech ("read answer aloud")
  12. Dark / light mode toggle

Run with:
    python app.py
"""

import json
import tempfile

import gradio as gr
import pandas as pd

from chatbot import FAQChatbot, load_faqs, CATEGORIES, FALLBACK_MESSAGE, HUMAN_SUPPORT_MESSAGE
from voice_utils import transcribe_audio, synthesize_speech

FAQ_PATH = "faq.csv"
ADMIN_PASSWORD = "admin123"  # demo only — change before real deployment
MAX_SUGGESTIONS = 6

WELCOME = "👋 Hi! Ask me anything about admissions, fees, programs, exams, hostel, library, or any other university service."

# ---------------------------------------------------------------------- #
# Shared chatbot instance, trained on the full 1000-question dataset.
# ---------------------------------------------------------------------- #
bot = FAQChatbot(load_faqs(FAQ_PATH))


# ---------------------------------------------------------------------- #
# Chat logic
# ---------------------------------------------------------------------- #
def format_bot_reply(result):
    emoji = "💬" if result["is_confident"] else "🤔"
    pct = int(round(result["score"] * 100))
    cat = f" _(Category: {result['category']})_" if result.get("category") else ""
    note = f"_Answer confidence: {pct}%_{cat}" if result["is_confident"] else f"_{HUMAN_SUPPORT_MESSAGE}_"
    return f"{emoji} {result['answer']}\n\n{note}"


def ask_question(message, history, category, tts_enabled):
    history = history or []
    if not message or not message.strip():
        return history, "", None

    result = bot.get_best_match(message, category=category)
    reply = format_bot_reply(result)

    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": reply},
    ]

    audio_path = None
    if tts_enabled:
        audio_bytes, err = synthesize_speech(result["answer"], language="en")
        if audio_bytes:
            tmp = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False)
            tmp.write(audio_bytes)
            tmp.close()
            audio_path = tmp.name
        elif err:
            gr.Warning(err)

    return history, "", audio_path


def ask_via_voice(audio_path, history, category, tts_enabled):
    if not audio_path:
        gr.Warning("Please record a question first.")
        return history, None, None
    text, err = transcribe_audio(audio_path, language="en")
    if err:
        gr.Warning(err)
        return history, None, None
    history, _, tts_audio = ask_question(text, history, category, tts_enabled)
    return history, None, tts_audio


def clear_chat():
    return [{"role": "assistant", "content": WELCOME}], None


# ---------------------------------------------------------------------- #
# Suggested questions
# ---------------------------------------------------------------------- #
def get_filtered_questions(category):
    items = [item for item in bot.faqs if category == "All" or item["category"] == category]
    return items[:MAX_SUGGESTIONS]


def refresh_suggestion_buttons(category):
    items = get_filtered_questions(category)
    updates = []
    for i in range(MAX_SUGGESTIONS):
        if i < len(items):
            q_text = items[i]["question"]
            updates.append(gr.update(value=f"• {q_text}", visible=True))
        else:
            updates.append(gr.update(value="", visible=False))
    return updates


def suggestion_clicked(btn_index, category):
    items = get_filtered_questions(category)
    if btn_index < len(items):
        return items[btn_index]["question"]
    return ""


# ---------------------------------------------------------------------- #
# Admin panel logic
# ---------------------------------------------------------------------- #
def faqs_to_dataframe():
    df = pd.DataFrame(bot.faqs)[["id", "category", "question", "answer"]]
    df.columns = ["ID", "Category", "Question", "Answer"]
    return df


def admin_login(password):
    if password == ADMIN_PASSWORD:
        choices = faq_choices()
        return (
            gr.update(visible=False),
            gr.update(visible=True),
            "✅ Logged in.",
            faqs_to_dataframe(),
            gr.update(choices=choices, value=(choices[0][1] if choices else None)),
        )
    return (
        gr.update(visible=True),
        gr.update(visible=False),
        "❌ Incorrect password.",
        faqs_to_dataframe(),
        gr.update(),
    )


def faq_choices():
    return [(f"#{item['id']} — {item['question']}", item["id"]) for item in bot.faqs]


def search_faqs(keyword):
    if not keyword or not keyword.strip():
        return faqs_to_dataframe()
    kw = keyword.strip().lower()
    filtered = [
        item for item in bot.faqs
        if kw in item["question"].lower() or kw in item["answer"].lower() or kw in item["category"].lower()
    ]
    if not filtered:
        return pd.DataFrame(columns=["ID", "Category", "Question", "Answer"])
    df = pd.DataFrame(filtered)[["id", "category", "question", "answer"]]
    df.columns = ["ID", "Category", "Question", "Answer"]
    return df


def add_faq(category, question, answer):
    if not question or not answer:
        gr.Warning("Question and answer are both required.")
        return faqs_to_dataframe(), gr.update(choices=faq_choices()), question, answer
    bot.add_faq(category, question, answer)
    gr.Info("FAQ added.")
    choices = faq_choices()
    return faqs_to_dataframe(), gr.update(choices=choices, value=choices[-1][1] if choices else None), "", ""


def load_faq_for_edit(faq_id):
    item = next((i for i in bot.faqs if i["id"] == faq_id), None)
    if not item:
        return CATEGORIES[0], "", ""
    return item["category"], item["question"], item["answer"]


def save_faq(faq_id, category, question, answer):
    if faq_id is None:
        gr.Warning("No FAQ selected.")
        return faqs_to_dataframe(), gr.update(choices=faq_choices())
    bot.update_faq(faq_id, category=category, question=question, answer=answer)
    gr.Info("FAQ updated.")
    return faqs_to_dataframe(), gr.update(choices=faq_choices(), value=faq_id)


def delete_faq(faq_id):
    if faq_id is None:
        gr.Warning("No FAQ selected.")
        return faqs_to_dataframe(), gr.update(choices=faq_choices()), CATEGORIES[0], "", ""
    bot.delete_faq(faq_id)
    gr.Info("FAQ deleted.")
    choices = faq_choices()
    new_value = choices[0][1] if choices else None
    cat, q, a = load_faq_for_edit(new_value) if new_value else (CATEGORIES[0], "", "")
    return faqs_to_dataframe(), gr.update(choices=choices, value=new_value), cat, q, a


def save_to_disk():
    bot.save_to_csv(FAQ_PATH)
    gr.Info(f"Saved {len(bot.faqs)} FAQs to {FAQ_PATH}")


def export_csv():
    tmp = tempfile.NamedTemporaryFile(suffix=".csv", delete=False, mode="w", encoding="utf-8", newline="")
    pd.DataFrame(bot.faqs)[["id", "category", "question", "answer"]].to_csv(tmp.name, index=False)
    return tmp.name


# ---------------------------------------------------------------------- #
# UI
# ---------------------------------------------------------------------- #
DARK_MODE_JS = "() => { document.body.classList.toggle('dark'); }"

with gr.Blocks(title="Smart University FAQ Assistant") as demo:
    gr.Markdown("# 🎓 Smart University AI FAQ Chatbot")
    gr.Markdown(
        f"Trained on **{len(bot.faqs)} questions** across **{len(CATEGORIES)} categories**. "
        "Ask in your own words — matched using **TF-IDF + cosine similarity**."
    )

    with gr.Tabs():
        # ================================================================ #
        # CHAT TAB
        # ================================================================ #
        with gr.Tab("💬 Chat"):
            with gr.Row():
                tts_enabled = gr.Checkbox(label="🔊 Read answers aloud", value=False, scale=1)
                dark_toggle = gr.Button("🌙 Toggle dark mode", scale=1)

            with gr.Row():
                with gr.Column(scale=3):
                    chatbot_ui = gr.Chatbot(
                        value=[{"role": "assistant", "content": WELCOME}],
                        height=420, label="Chat",
                    )
                    tts_audio = gr.Audio(autoplay=True, visible=True, label="Bot voice", show_label=False)

                    with gr.Row():
                        msg_box = gr.Textbox(
                            placeholder="Type your question here...", scale=4,
                            show_label=False, container=False,
                        )
                        send_btn = gr.Button("Ask", variant="primary", scale=1)

                    with gr.Accordion("🎤 Or ask by voice", open=False):
                        mic_input = gr.Audio(sources=["microphone"], type="filepath", label="Record your question")
                        voice_ask_btn = gr.Button("Transcribe & Ask")

                    clear_btn = gr.Button("🗑️ Clear chat")

                with gr.Column(scale=1):
                    gr.Markdown("### 📂 Category")
                    category_filter = gr.Dropdown(
                        choices=["All"] + CATEGORIES, value="All", label="Filter suggested questions"
                    )
                    gr.Markdown("### 💡 You can ask")
                    suggestion_buttons = [
                        gr.Button("", visible=False, size="sm") for _ in range(MAX_SUGGESTIONS)
                    ]

            # --- wiring ---
            send_btn.click(
                ask_question, [msg_box, chatbot_ui, category_filter, tts_enabled],
                [chatbot_ui, msg_box, tts_audio],
            )
            msg_box.submit(
                ask_question, [msg_box, chatbot_ui, category_filter, tts_enabled],
                [chatbot_ui, msg_box, tts_audio],
            )
            voice_ask_btn.click(
                ask_via_voice, [mic_input, chatbot_ui, category_filter, tts_enabled],
                [chatbot_ui, mic_input, tts_audio],
            )
            clear_btn.click(clear_chat, None, [chatbot_ui, tts_audio])
            dark_toggle.click(fn=None, js=DARK_MODE_JS)

            category_filter.change(refresh_suggestion_buttons, [category_filter], suggestion_buttons)
            demo.load(refresh_suggestion_buttons, [category_filter], suggestion_buttons)
            for idx, btn in enumerate(suggestion_buttons):
                btn.click(
                    lambda category, i=idx: suggestion_clicked(i, category),
                    [category_filter], msg_box,
                ).then(
                    ask_question, [msg_box, chatbot_ui, category_filter, tts_enabled],
                    [chatbot_ui, msg_box, tts_audio],
                )

        # ================================================================ #
        # ADMIN TAB
        # ================================================================ #
        with gr.Tab("🛠️ Admin Panel"):
            gr.Markdown("Log in to manage FAQs. (Demo password: `admin123`)")

            with gr.Row(visible=True) as login_row:
                admin_pwd = gr.Textbox(label="Password", type="password", scale=3)
                login_btn = gr.Button("Log in", scale=1)
            login_status = gr.Markdown("")

            with gr.Column(visible=False) as admin_content:
                gr.Markdown(f"### 📋 All FAQs ({len(bot.faqs)} total)")
                search_box = gr.Textbox(label="🔎 Search by keyword or category", placeholder="e.g. hostel, fee, exam...")
                faq_table = gr.Dataframe(value=faqs_to_dataframe(), interactive=False, wrap=True, max_height=350)

                gr.Markdown("### ➕ Add a new FAQ")
                add_category = gr.Dropdown(choices=CATEGORIES, value=CATEGORIES[0], label="Category")
                add_question = gr.Textbox(label="Question")
                add_answer = gr.Textbox(label="Answer", lines=2)
                add_btn = gr.Button("➕ Add FAQ", variant="primary")

                gr.Markdown("### ✏️ Edit or delete an FAQ")
                edit_select = gr.Dropdown(choices=faq_choices(), label="Select FAQ", type="value")
                edit_category = gr.Dropdown(choices=CATEGORIES, value=CATEGORIES[0], label="Category")
                edit_question = gr.Textbox(label="Question")
                edit_answer = gr.Textbox(label="Answer", lines=2)
                with gr.Row():
                    save_btn = gr.Button("💾 Save changes", variant="primary")
                    delete_btn = gr.Button("🗑️ Delete FAQ", variant="stop")

                gr.Markdown("### 💾 Persist changes")
                with gr.Row():
                    save_disk_btn = gr.Button("Save to faq.csv on server")
                    download_btn = gr.DownloadButton("⬇️ Download faq.csv")

            # --- wiring ---
            login_btn.click(
                admin_login, [admin_pwd],
                [login_row, admin_content, login_status, faq_table, edit_select],
            )

            search_box.change(search_faqs, [search_box], [faq_table])

            add_btn.click(
                add_faq, [add_category, add_question, add_answer],
                [faq_table, edit_select, add_question, add_answer],
            )

            edit_select.change(
                load_faq_for_edit, [edit_select],
                [edit_category, edit_question, edit_answer],
            )

            save_btn.click(
                save_faq, [edit_select, edit_category, edit_question, edit_answer],
                [faq_table, edit_select],
            )

            delete_btn.click(
                delete_faq, [edit_select],
                [faq_table, edit_select, edit_category, edit_question, edit_answer],
            )

            save_disk_btn.click(save_to_disk, None, None)
            download_btn.click(export_csv, None, download_btn)


if __name__ == "__main__":
    demo.queue()
    demo.launch(theme=gr.themes.Soft())


## 3. Quick sanity check (no GUI yet)

Import the chatbot engine directly and try a few paraphrased questions —
useful for confirming the matching logic before launching the full GUI.


In [ ]:
import sys
sys.path.insert(0, ".")
import importlib
import chatbot
importlib.reload(chatbot)

bot_test = chatbot.FAQChatbot(chatbot.load_faqs("faq.csv"))
print(f"Loaded {len(bot_test.faqs)} FAQs across {len(set(f['category'] for f in bot_test.faqs))} categories")

test_questions = [
    "What do I need to get admitted into BS Computer Science?",
    "How much does the Computer Science degree cost?",
    "when do midterm exams begin",
    "I can't remember my student portal password",
    "how many books can I take from the library",
]

for q in test_questions:
    result = bot_test.get_best_match(q)
    print(f"\nUser: {q}")
    print(f"  Matched [{result['category']}]: {result['matched_question']}")
    print(f"  Score: {result['score']:.3f}  |  Confident: {result['is_confident']}")
    print(f"  Bot: {result['answer']}")


## 4. Launch the Gradio app

This imports `app.py` (which builds the full Chat + Admin Panel UI) and
launches it with `share=True` so Colab gives you a public URL — click the
link that gets printed below to open the GUI in a new tab.

**Demo admin password:** `admin123`


In [ ]:
import importlib
import app
importlib.reload(app)

app.demo.queue()
app.demo.launch(share=True, debug=False)


## Notes / limitations

- **Retrieval-based, not generative** — the bot only returns answers that
  already exist in `faq.csv`. No hallucination risk, but it can't invent new
  answers outside the dataset.
- **Two real issues were found and fixed while tuning this on your actual
  data:**
  1. NLTK's lemmatizer was reducing "BS" to "B" (treating it as a plural
     noun), which hurt matching on every program name — fixed by protecting
     abbreviations (`bs`, `ms`, `mba`, `bba`, `phd`, `gpa`, `cgpa`, `id`)
     from lemmatization.
  2. Domain synonyms (`admitted → admission`, `tuition/cost → fee`,
     `professor/instructor → faculty`, etc.) were added so paraphrases land
     in the right category. See `FAQChatbot.SYNONYMS` in `chatbot.py` if you
     want to extend this list.
- **Some ambiguity is inherent to this dataset** — many rows are templated
  near-duplicates (e.g. several worded variants of "how long is the BS CS
  program?"), and categories like Fees/Programs share a lot of vocabulary.
  Use the category filter in the Chat tab to disambiguate when a question is
  vague, or swap in sentence embeddings for stronger semantic matching if
  you outgrow TF-IDF.
- **Confidence score** is the cosine similarity score (0–100%) — a genuine
  relative match-quality signal, not a calibrated probability.
- **Voice input and text-to-speech need internet** — they call free Google
  endpoints (`SpeechRecognition`'s Google Web Speech API, and `gTTS`). Both
  are wrapped in try/except so the app shows a warning instead of crashing
  if there's no connection or mic access.
- **`share=True` is required in Colab** — Colab can't reach `localhost`
  directly, so Gradio creates a temporary public tunnel URL instead. Don't
  put anything sensitive in the admin panel while it's running this way.
- **Admin password is a hardcoded demo value** (`admin123`) — fine for a
  portfolio project, not real authentication.
- To stop the app, interrupt the cell (■ button) or restart the runtime.
- To reuse this outside Colab, download `app.py`, `chatbot.py`,
  `voice_utils.py`, and `faq.csv` from the Colab file browser (left sidebar)
  and run `python app.py` locally — no code changes needed.
